`CELDA 1`

# Fase 3 · Preparación de los datos · SECOP II

**Metodología:** CRISP-DM — tercera fase
**Fuente:** Portal de Datos Abiertos de Colombia — Colombia Compra Eficiente
**Fecha de corte del proyecto:** 21 de septiembre de 2026
**Equipo:** Miguel Luna · Luisa Portela · Santiago

---

## Objetivo de la fase

Convertir cuatro bases publicadas por separado en **un único conjunto de datos
listo para modelar**: con una variable objetivo definida, variables explicativas
construidas y una verificación que certifique que el paso 4 puede empezar sin
volver atrás.

## Enfoque

| Elemento | Definición |
|---|---|
| **Usuario receptor** | La empresa proveedora que decide a qué proceso presentarse |
| **Componente de control** | Riesgo de la entidad contratante, incorporado como variables de contexto |
| **Unidad de análisis** | El proceso de contratación |
| **Variable objetivo** | `adjudicado` · 1 si el proceso terminó adjudicado, 0 si quedó desierto o se canceló |

## La pregunta que responde

Preparar una oferta cuesta tiempo y dinero. Un proceso que termina desierto consume
ese esfuerzo y no devuelve nada. La pregunta operativa es entonces doble: dónde hay
adjudicación probable, y qué entidades concentran los procesos que no llegan a
término.

## Las cuatro bases

| Base | Código en el portal | Filas traídas | Peso | Papel |
|---|---|---:|---:|---|
| Procesos de Contratación | `p6dx-8zbt` | 8.516.193 | 2,94 GB | Tabla principal y variable objetivo |
| Contratos Electrónicos | `jbjy-vk9h` | 5.334.841 | 2,26 GB | Ejecución y pago de la entidad |
| Proveedores Registrados | `qmzu-gj57` | 1.618.521 | 0,24 GB | Perfil del proveedor |
| Grupos de Proveedores | `ceth-n4bn` | 711.140 | 0,15 GB | Experiencia en consorcios |

En total **16,2 millones de registros y 5,58 GB**, casi cuatro veces el mínimo de
1,5 GB exigido al proyecto.

Los datos se traen directamente del portal por su interfaz de consulta, no por
descarga manual. Eso hace el trabajo repetible: ejecutar este cuaderno produce
exactamente los mismos datos, con los mismos filtros y las mismas columnas.

## Contenido

| Paso | Tarea | Qué se hace |
|---|---|---|
| 1 | Traer los datos | Conexión al portal y descarga por tandas |
| 2 | Seleccionar y convertir | Columnas, tipos y formatos |
| 3 | Limpiar | Los defectos de la fuente |
| 4 | Depurar | Nulos, duplicados y valores atípicos |
| 5 | Construir | Variable objetivo y variables derivadas |
| 6 | Integrar | Unión de las cuatro bases |
| 7 | Agregar y derivar | Transformación analítica y conjuntos derivados |
| 8 | Probar con Dask | Configuración A (2 trabajadores × 2 GB) contra B (4 × 4 GB) |
| 9 | Orquestar con Prefect | Flujo Extract → Transform → Validate → Analyze → Export |
| 10 | Verificar | Puerta de calidad y continuidad hacia el paso 4 |

## Lo que esta fase entrega al paso 4

La fase siguiente debe poder arrancar sin descargar nada ni ejecutar ninguno de los
pasos de aquí. Para eso se producen cinco archivos:

| Entregable | Para qué sirve |
|---|---|
| `tabla_analitica.parquet` | El conjunto derivado, con los tipos conservados |
| `diccionario_fase3.csv` | Qué significa cada variable, de dónde viene y qué tan completa está |
| `reporte_calidad.csv` | El resultado de la puerta de calidad de esta corrida |
| `banco_dask.csv` | La comparación de las dos configuraciones de cómputo |
| `flujo_prefect.py` | El flujo ETL, repetible con datos nuevos |

`CELDA 2`

## Paso 0 · Por qué la tabla se arma así

Conviene dejar anotado desde el comienzo por qué entra cada base. Sin esa nota, en
dos semanas nadie recuerda por qué una tabla quedó adentro y otra afuera, y la
decisión queda sin respaldo.

Son cuatro bases y cada una entra por un motivo distinto.

**Procesos de Contratación** es la principal. Es la única que indica si un proceso
terminó adjudicado o se quedó sin adjudicar, que es justo lo que se quiere predecir.
Las demás giran alrededor de esta.

**Contratos Electrónicos** muestra qué pasó después de la firma: cuánto se le pagó
al proveedor, cuánto quedó debiendo la entidad y cuántos días se alargó el contrato.
Sirve para caracterizar qué tipo de cliente es cada entidad.

**Proveedores Registrados** es la lista de las empresas habilitadas para venderle al
Estado. De ahí salen la antigüedad de cada proveedor, su tamaño, si es pyme y dónde
está ubicado. Como el análisis está planteado desde la empresa que quiere licitar,
esta es la base que describe al sujeto del estudio.

**Grupos de Proveedores** muestra qué empresas se asocian entre sí para presentarse
juntas. Ninguna otra base contiene esa información, y en procesos grandes importa,
porque con frecuencia no gana una empresa sola sino un consorcio.

La arquitectura queda así:

```
       PROCESOS  ──────────────────  la tabla principal: una fila por proceso
          │                          (8,5 millones de filas)
          ├── CONTRATOS     ── cómo paga y cómo cumple la entidad
          ├── PROVEEDORES   ── quién es el proveedor y desde cuándo existe
          └── GRUPOS        ── con quién se asocia para competir
```

### La parte de control

El proyecto está enfocado en la empresa que quiere licitar, con elementos que sirvan
también a quien vigila la contratación. Esa segunda mirada no va en un capítulo
aparte: entra como tres variables que describen a la entidad que convoca.

- **Cuántos días se alargan sus contratos.** Un contrato que se prorroga mucho suele
  venir de una planeación deficiente.
- **Cuánto se desvía lo facturado de lo pactado.** Si una entidad firma por un valor
  y termina facturando otro muy distinto, algo ocurrió en el camino.
- **Qué proporción de sus procesos deja desiertos.** Una entidad que convoca y no
  adjudica le hace perder a la empresa el trabajo de preparar la propuesta.

Las tres le hablan a los dos lectores a la vez. A la empresa le indican con qué
entidades conviene involucrarse; a quien controla, dónde mirar primero.

El límite hay que declararlo: estas variables miden la prórroga en tiempo y la
desviación en dinero, no el número de modificaciones formales que se le hicieron al
contrato. Ese dato está en otra base que no forma parte de este conjunto.

### Tres restricciones que condicionan el resto del trabajo

**El tamaño.** Las dos bases grandes suman más de 30 GB en el portal. No caben en la
memoria de un equipo de escritorio, así que no se pueden traer enteras. Por eso el
paso 1 pide solo las columnas necesarias y lo hace por tandas.

**Los datos personales.** Estas bases incluyen información de personas que no tiene
relación con el análisis:

| Base | Columnas que trae | Columnas personales |
|---|---|---|
| Proveedores Registrados | 26 | 10 — teléfono, fax, correo, dirección, página web, y nombre, cédula, teléfono y correo del representante legal |
| Grupos de Proveedores | 33 | 10 — las mismas, referidas al grupo |
| Contratos Electrónicos | 92 | 15 — cédula y domicilio del representante legal, género, banco, tipo y **número de cuenta bancaria**, y documentos de supervisores |

Esas columnas no se piden. Al traer los datos por consulta se escoge qué columnas
llegan, de modo que las personales nunca entran al equipo. Es más seguro que
traerlas y borrarlas después: lo que nunca llegó no puede filtrarse por accidente a
un entregable, y este trabajo circula entre varias personas y queda en un
repositorio.

**La fecha de corte.** Las cuatro bases se consultan con el mismo tope superior, el
**21 de septiembre de 2026**.

Parece que se estuviera descartando información, y así es, pero no hacerlo sale
peor. Comparar una entidad medida hasta octubre con otra medida hasta septiembre
hace que la primera aparezca con más procesos simplemente porque se la observó
durante más tiempo. Esa diferencia no dice nada del mundo real, solo del calendario.

### Organización de los archivos

Lo que se descarga pesa mucho; lo que produce esta fase pesa una fracción mínima de
eso. Esa diferencia define cómo se reparten las carpetas:

```
fase3_secop/                      ← carpeta que se comparte con el equipo
├── 03_preparacion_datos_v1.ipynb
├── flujo_prefect.py
├── salida_fase3/                 ← también se comparte: es el resultado
│   ├── tabla_analitica.parquet
│   ├── diccionario_fase3.csv
│   ├── reporte_calidad.csv
│   └── banco_dask.csv
└── datos/                        ← no se comparte: son las descargas del portal
```

Con esa carpeta, la fase siguiente abre el parquet y arranca sin descargar nada ni
repetir ninguno de los pasos de aquí. Esa independencia es un resultado de la fase y
no un detalle de orden: una preparación que obliga a reprocesar decenas de gigabytes
para poder continuar no sirve en la práctica, por bien hecha que esté.

`CELDA 3`

## Bitácora · qué cambió durante la preparación y por qué

Esta fase no salió derecha a la primera. Cada decisión que se ve en el código
vino de una medición que mostró que lo anterior no servía, y dejarlas anotadas
es parte del entregable: sin esto, el cuaderno parecería que adivinó bien de
entrada, y no fue así.

Las vueltas, en el orden en que ocurrieron.

---

### 1 · Los archivos descargados a mano no servían

El primer intento fue con dos archivos comprimidos de unos 15 GB bajados desde
el navegador. Los dos salieron corruptos: el comienzo del archivo estaba bien
pero el índice del final no existía, que es lo que pasa cuando una descarga se
corta y el navegador la da por terminada igual.

**Qué cambió:** todo se baja por la API del portal, mes por mes. Un mes que
falle se vuelve a pedir solo, y una interrupción no obliga a empezar de cero.
De paso, esto es lo que convierte la descarga en la etapa *Extract* del flujo
ETL, que es uno de los entregables.

---

### 2 · El rango empezaba en 2020 y se corrió a 2017

Al principio se fijó enero de 2020 como inicio, pensando en no descargar de
más. Al medir qué porcentaje de los procesos de cada año era competitivo
apareció lo contrario de lo esperado:

| año | procesos competitivos |
|---|---|
| 2018 | 38,0 % |
| 2024 | 10,6 % |

Los años viejos son proporcionalmente mucho más ricos en lo que interesa. Los
años recientes están dominados por contratación directa, que no sirve para este
análisis porque ahí no hay a quién ganarle.

**Qué cambió:** el inicio se corrió a enero de 2017. Eso sumó 102.344 procesos
competitivos, un 16,3 % más de lo que había. No era un 5 % como se estimó
primero; la estimación estaba mal y la medición la corrigió.

---

### 3 · No existe una tabla de ofertas

La idea original era tener una fila por cada vez que una empresa se presenta a
un proceso, que es la unidad natural para responder "¿me conviene presentarme?".
Esa tabla no existe en ninguna de las cuatro bases: SECOP II publica cuántos
proveedores se presentaron, pero no quiénes.

**Qué cambió:** la unidad de análisis pasó a ser el **proceso**, y la pregunta
pasó a ser "¿este proceso termina en adjudicación?". No es la pregunta ideal,
pero es la que los datos permiten responder sin inventar.

---

### 4 · Una fila por proceso no era una fila por proceso

Al contar identificadores repetidos apareció que un mismo proceso generaba
hasta 1.600 filas. Revisando qué columnas cambiaban entre esas filas se vio el
patrón: la fuente guarda el **producto cruzado** de 40 proveedores por 40
valores de adjudicación, no 1.600 adjudicaciones reales.

**Qué cambió:** se agregó un tercer filtro que deja una sola fila por proceso,
ordenando primero por el NIT del adjudicatario para no quedarse con la fila
vacía cuando hay una llena. De 9.069.883 procesos descargados quedaron 626.384,
que es el 6,9 %. Parece poco y es correcto: lo que se botó eran copias y
modalidades sin competencia.

---

### 5 · Se agregó Contratos como base de cruce

La tabla de Procesos deja el NIT del ganador sin llenar en más de la mitad de
los casos. Con solo esa columna, la cobertura del adjudicatario era 38,9 %.

**Qué cambió:** se agregó Contratos. Un proceso que terminó en contrato firmado
tiene ahí al proveedor, aunque la casilla de Procesos esté vacía. La cobertura
subió a **82,9 %**.

---

### 6 · El cruce entre Procesos y Contratos no funcionaba

El primer intento de unir las dos tablas devolvió cero filas. La causa: SECOP II
le da **dos identificadores distintos al mismo proceso**. En Procesos se llama
`id_del_proceso` y empieza por `CO1.REQ`; en Contratos se llama
`proceso_de_compra` y empieza por `CO1.BDOS`. No son el mismo texto.

La única columna que los relaciona es `id_del_portafolio`, y no se había pedido
en la descarga original porque nadie sabía que haría falta.

**Qué cambió:** se agregó una descarga aparte de dos columnas, solo para las
modalidades competitivas. Son 1.235.403 parejas y toma unos cuatro minutos.

---

### 7 · Dos columnas se habían descartado por error

Al armar la lista de columnas que se le piden al portal se dejaron por fuera
dos que después hubo que recuperar:

- `nit_del_proveedor_adjudicado`, descartada por estar vacía el 93 % de las
  veces. Ese porcentaje estaba inflado por la contratación directa; dentro de
  las modalidades competitivas la columna sí tiene contenido.
- `id_del_portafolio`, que resultó ser la única llave del cruce.

**Qué cambió:** las dos volvieron a la lista. La lección quedó anotada: un
porcentaje de vacíos medido sobre toda la base puede estar diciendo algo
distinto de lo que parece.

---

### 8 · La variable objetivo estaba mal definida

Se iba a usar directamente la casilla `adjudicado` de la fuente. Al cruzarla
contra la evidencia de contratos firmados aparecieron **22.151 procesos**
marcados como no adjudicados que sí tenían contrato.

**Qué cambió:** la variable objetivo se construye combinando las dos señales.
Pasó de 82,5 % a **86,0 %** de procesos adjudicados.

---

### 9 · Las variables de pago no medían pagos

El perfil de entidad incluía la proporción pagada sobre lo contratado. Al
medirla por año de firma del contrato salió así:

| año de firma | contratos con pago registrado |
|---|---|
| 2018 | 0,0 % |
| 2024 | 37,2 % |

Un contrato de 2018 tuvo seis años más para pagarse que uno de 2024, así que la
curva debería ir al revés. Que vaya en este sentido significa que la columna no
mide si se pagó, mide **si alguien registró el pago en la plataforma**, y ese
registro es una práctica reciente.

**Qué cambió:** la variable se renombró a `ent_reporta_pagos_pct` y se dejó
descrita como hábito administrativo, no como cumplimiento. Se botaron las otras
dos variables de pago.

---

### 10 · El perfil de entidad le estaba contando la respuesta al modelo

La primera versión calculaba la tasa de adjudicación de cada entidad usando
todos sus procesos, incluido el que se quiere predecir. Eso es fuga de
información: el modelo ve el resultado metido dentro de una de sus variables de
entrada.

**Qué cambió:** el perfil se calcula con ventana hacia atrás. Un proceso de 2021
recibe el perfil armado con los procesos de 2017 a 2020 de esa misma entidad, y
nada más. Después de corregirlo la variable **sigue separando**: los procesos de
entidades con tasa histórica baja se adjudican el 55,0 % de las veces, y los de
entidades con tasa alta, el 90,0 %. Treinta y cinco puntos de diferencia, ahora
sí sin trampa.

---

### 11 · Los recortes por percentil no recortaban nada

Las cifras absurdas se iban a acotar con el percentil 99,5 de cada columna. Para
`ent_valor_medio` ese percentil era él mismo una cifra absurda —
40.535.792.367.663.624 pesos — y el recorte dejó pasar todo.

La razón: una entidad con miles de procesos y un valor mal digitado arrastra su
propio percentil.

**Qué cambió:** se pasó a topes absolutos, calibrados contra una referencia
real. El presupuesto general de la Nación ronda los 500 billones de pesos al
año, así que un contrato promedio de más de un billón no existe. Con ese tope,
el recorte sí alcanzó 12.075 filas.

---

### 12 · Dos variables derivadas estaban rotas

- `interes_por_vista` salía en cero en todas las filas. Depende de
  `proveedores_que_manifestaron`, que está en cero el 100 % de las veces: la
  casilla existe en el formulario pero no se diligencia.
- `duracion_dias` llegaba a 902.460.750 días, que son dos millones y medio de
  años, y `dias_para_cerrar` bajaba a −731 días.

**Qué cambió:** la primera se eliminó junto con la columna que la alimentaba.
Las otras dos se acotaron a rangos posibles, dejando vacío lo que no se puede
afirmar en vez de inventarle un cero.

---

### 13 · Una variable que parecía observable no lo era

`visualizaciones_del` está bien poblada y parecía una buena variable: cuánta
gente miró el proceso. El número que guarda la fuente es el total **al
cerrarse**, no el del día de la publicación, que es cero para todos.

Un modelo entrenado con esa columna aprendería que los procesos muy vistos se
adjudican. Es cierto y es inútil: la empresa que está decidiendo si se presenta
no puede saber cuántas visitas tendrá el proceso dentro de un mes.

**Qué cambió:** pasó de clase B a clase C. Es el tipo de fuga más difícil de
detectar, porque la variable no parece venir del futuro.

---

### 14 · Se agregaron Proveedores y Grupos para medir la competencia

Las dos bases que describen al proveedor no entraban en ninguna parte. Usarlas
para armar un perfil del ganador no sirve: el ganador se conoce al cerrar, así
que cualquier variable construida sobre él es clase C.

**Qué cambió:** se usan para contar cuántos proveedores estaban inscritos en el
departamento y en la categoría del proceso **antes** de que se publicara. Ese
número sí se conoce el día de la publicación, y responde directo a la pregunta
del proyecto: contra cuántos compito.

---

### Lo que queda declarado como limitación

- 4.099 procesos (0,65 %) comparten identificador de portafolio con otro
  proceso, así que un contrato podría quedar atribuido al proceso equivocado.
- Las cuatro variables de entidad que vienen de Contratos no están partidas por
  año: para un proceso de 2018, el conteo de contratos de su entidad incluye
  contratos posteriores. El riesgo es bajo porque describen tamaño y hábito
  administrativo, no el desenlace de un proceso en particular.
- Los procesos de 2017 no tienen perfil de entidad, porque no hay año anterior
  con el cual construirlo.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 4  ·  Entorno, rutas y constantes
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Nada. Esta celda no toca ningún dato todavía.
# QUÉ HACE   : Carga las herramientas del cuaderno y deja fijadas las
#              decisiones que no pueden cambiar a mitad del trabajo: de dónde
#              se traen los datos, hasta qué fecha se cuenta, desde cuándo, y
#              qué textos significan en realidad "aquí no hay dato".
# QUÉ SALE   : Un resumen en pantalla de esas decisiones, y las carpetas de
#              trabajo creadas.
# PARA QUÉ   : Para que todo lo demás del cuaderno lea de aquí. Un cambio de
#              fecha o de ruta se hace en esta celda y en ninguna otra.
# ──────────────────────────────────────────────────────────────────────

import gc
import json
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt


# ── De dónde vienen los datos ─────────────────────────────────────────
# El portal de datos abiertos publica cada base con un código corto. Ese
# código es el que va en la dirección de consulta.
PORTAL = "https://www.datos.gov.co/resource"

BASES = {
    "procesos":    {"codigo": "p6dx-8zbt", "fecha": "fecha_de_publicacion_del"},
    "contratos":   {"codigo": "jbjy-vk9h", "fecha": "fecha_de_firma"},
    "proveedores": {"codigo": "qmzu-gj57", "fecha": "fecha_creacion"},
    "grupos":      {"codigo": "ceth-n4bn", "fecha": "fecha_creaci_n_grupo"},
}


# ── Dónde se guarda todo ──────────────────────────────────────────────
# Lo descargado va en "datos". Lo que produce la fase va en "salida". Esa
# separación importa: la carpeta de salida es la que se comparte con el
# equipo, la de datos no, porque pesa demasiado.
DATOS  = Path("datos")
SALIDA = Path("salida_fase3")
DATOS.mkdir(exist_ok=True)
SALIDA.mkdir(exist_ok=True)


# ── Hasta cuándo y desde cuándo ───────────────────────────────────────
# Tope superior: vale para las cuatro bases. Medir una hasta octubre y otra
# hasta septiembre haría que la primera apareciera con más registros solo por
# haber sido observada durante más tiempo.
CORTE = pd.Timestamp("2026-09-21")

# Tope inferior: vale solo para las dos bases que registran hechos con fecha.
# Arranca en 2017 porque de ahí en adelante hay volumen suficiente de procesos
# competitivos, que son los únicos a los que una empresa puede presentarse;
# 2015 y 2016 aportan menos de seis mil y son los años más afectados por la
# adopción voluntaria de la plataforma. El razonamiento completo, con las
# cifras medidas, está en el paso 1.6.
#
# Las listas de proveedores y de grupos entran completas, porque una empresa
# inscrita en 2016 que hoy gana contratos tiene que estar presente.
INICIO = pd.Timestamp("2017-01-01")

CON_TOPE_INFERIOR = ("procesos", "contratos")


# ── Los textos que significan "vacío" ─────────────────────────────────
# Estas bases nunca dejan una celda en blanco: cuando no tienen el dato,
# escriben "No Definido" o algo equivalente. Para el programa eso es un texto
# normal, así que un conteo de vacíos daría cero y la base parecería impecable.
#
# La lista va en minúscula porque la fuente escribe lo mismo de varias formas
# ("No Definido", "No definido", "No DEFINIDO"), y comparar todo en minúscula
# es lo que hace que las tres caigan en el mismo saco.
VACIOS = {
    "no definido", "no definida", "no provisto", "no aplica", "no adjudicado",
    "no aplica/no pertenece", "n/a", "na", "sin dato", "no informado",
    "-", ".", "null", "none", "0001-01-01t00:00:00.000",
}


# ── Cómo se ven las tablas y los gráficos ─────────────────────────────
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)

TINTA, AZUL, DORADO, REJILLA = "#1C1C1C", "#2E5C76", "#C98A2B", "#E3E3E3"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.titlesize": 12.5, "axes.labelcolor": TINTA,
                     "text.color": TINTA, "axes.edgecolor": "#BFBFBF"})


def limpiar_ejes(ax, eje_y=True):
    """Quita el marco superior y derecho de un gráfico.

    Esas dos líneas no aportan nada y ensucian la lectura. Deja una rejilla
    muy tenue de fondo para poder seguir los valores con la vista.
    """
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    if eje_y:
        ax.grid(axis="y", color=REJILLA, lw=.8)
        ax.set_axisbelow(True)


# ── Resumen de las decisiones ─────────────────────────────────────────
print(f"pandas {pd.__version__}   ·   numpy {np.__version__}\n")

print("DECISIONES DE ESTA FASE")
print("-" * 74)
print(f"  Cada fila es        : un proceso de contratación")
print(f"  Variable objetivo   : si el proceso se adjudicó o quedó desierto")
print(f"  Se cuenta hasta     : {CORTE:%d/%m/%Y}")
print(f"  Se cuenta desde     : {INICIO:%d/%m/%Y}  (solo en procesos y contratos)")
print(f"  Meses por base      : {len(pd.period_range(INICIO, CORTE, freq='M'))}")
print(f"  Carpeta de descarga : {DATOS.resolve()}")
print(f"  Carpeta de entrega  : {SALIDA.resolve()}")

print("\nBASES QUE SE VAN A CONSULTAR")
print("-" * 74)
for nombre, info in BASES.items():
    tope = f"desde {INICIO:%Y-%m-%d}" if nombre in CON_TOPE_INFERIOR else "completa"
    print(f"  {nombre:12} {info['codigo']:12} {tope:22} fecha: {info['fecha']}")

`CELDA 5`

## Paso 1 · Traer los datos del portal

### 1.1 Cómo se le pide información al portal

El portal publica una dirección de consulta por cada base. Agregando instrucciones
al final de esa dirección, devuelve justo lo pedido en vez de la base entera. Son
cuatro instrucciones y con esas cuatro se arma todo:

| Instrucción | Qué indica | Ejemplo |
|---|---|---|
| `$select` | Qué columnas | `$select=id_del_proceso,precio_base` |
| `$where` | Qué filas | `$where=precio_base > 1000000` |
| `$limit` | Cuántas filas por pedido | `$limit=50000` |
| `$offset` | Desde cuál fila empieza a contar | `$offset=100000` |

Las dos últimas son las que permiten traer una base grande: se piden las primeras
cincuenta mil, luego las cincuenta mil siguientes, y así sucesivamente. A eso se le
llama paginar, igual que pasar las hojas de un libro.

### 1.2 Por qué no se pide todo de una sola vez

El portal tiene un tope: no entrega más de unos cincuenta mil registros por pedido.
Pero aunque no lo tuviera, tampoco convendría. Una sola conexión pidiendo ocho
millones de filas, interrumpida en el minuto cuarenta, pierde los cuarenta minutos.
Pidiendo por tandas, lo que ya llegó queda guardado.

### 1.3 Por qué se baja mes por mes y no seguido

Hay un detalle del funcionamiento del portal que condiciona el diseño. Ante un
`$offset=4000000`, el portal tiene que recorrer cuatro millones de filas antes de
entregar la primera. Entre más avanza la descarga, más lento responde, hasta que
corta la conexión.

La forma de evitarlo es no dejar que ese número crezca. En lugar de pedir ocho
millones seguidos, se pide **un mes a la vez**. Cada mes trae entre cincuenta y
ciento setenta mil procesos, así que el contador nunca sube de ahí y la velocidad se
mantiene pareja de principio a fin.

De paso resuelve otra cosa: cada mes queda en su propio archivo. Si la descarga se
interrumpe en el mes cuarenta, al reanudarla se salta los treinta y nueve ya
guardados y continúa donde iba.

```
datos/
└── procesos/
    ├── procesos_2017-01.csv
    ├── procesos_2017-02.csv
    ├── procesos_2017-03.csv
    └── ...  un archivo por mes, 117 en total
```

### 1.4 Qué columnas se piden

De las 60 columnas de Procesos y las 92 de Contratos se pide menos de la mitad. Una
columna queda afuera si cumple alguna de estas tres condiciones:

**Es información personal.** Cédulas, direcciones, teléfonos, correos, cuentas
bancarias. No son variables del problema y no deben circular en un trabajo que se
comparte y se publica en un repositorio.

**Es texto libre muy largo.** La descripción del procedimiento pesa más que todas
las demás columnas juntas y no entra en ningún cálculo de esta fase. Dejarla afuera
reduce el tamaño de la descarga a una fracción.

**No aporta al análisis.** Códigos internos del sistema, direcciones de páginas web
y nombres que ya vienen repetidos en otra columna.

Sí se piden las columnas del resultado: si el proceso se adjudicó, a quién y por
cuánto. De ahí sale la variable objetivo. Quedan señaladas aparte porque son
información que solo se conoce una vez cerrado el proceso, y mezclarlas con las
variables explicativas haría que cualquier modelo acertara en el cuaderno y fallara
en la realidad.

### 1.5 Hasta dónde se trae cada base, y por qué no es igual para todas

Las cuatro bases no son del mismo tipo, y eso cambia cómo se recortan.

**Procesos y Contratos registran hechos.** Cada fila es algo que ocurrió en una
fecha concreta: un proceso que se publicó, un contrato que se firmó. Recortarlas por
tiempo tiene sentido, porque equivale a escoger el periodo de estudio.

**Proveedores y Grupos son listas de quién existe.** La fecha que traen es la de
inscripción, no la de una actuación. Recortarlas por tiempo no acota un periodo:
elimina empresas.

El error que esto evita conviene verlo concreto. Recortar la lista de proveedores a
los inscritos después de 2017 sacaría de la base a toda empresa registrada en 2016
que hoy está ganando contratos. Al cruzar las tablas, esos procesos quedarían sin
proveedor: no porque no lo tengan, sino por haberlo eliminado. El análisis mostraría
un vacío inexistente.

| Base | Desde | Hasta |
|---|---|---|
| Procesos | 1 de enero de 2017 | 21 de septiembre de 2026 |
| Contratos | 1 de enero de 2017 | 21 de septiembre de 2026 |
| Proveedores | completa | 21 de septiembre de 2026 |
| Grupos | completa | 21 de septiembre de 2026 |

El tope superior sí aplica a las cuatro por igual, y por una razón distinta: medir
una entidad hasta octubre y otra hasta septiembre inventa diferencias que vienen del
calendario y no del fenómeno.

### 1.6 Por qué el periodo arranca en 2017

El registro más antiguo de Procesos es de abril de 2015, así que se podría haber
traído todo. La decisión sale de contar, no el total de procesos por año, sino
**cuántos de ellos son competitivos**, que son los únicos a los que una empresa
puede presentarse.

Se midieron tres años contra el portal:

| Año | Procesos totales | Competitivos | Participación |
|---|---:|---:|---:|
| 2018 | 194.742 | 74.016 | 38,0 % |
| 2021 | 652.936 | 118.885 | 18,2 % |
| 2024 | 1.670.370 | 176.911 | 10,6 % |

Lo que muestran esos tres cortes es un cambio profundo en cómo contrata el Estado.
Entre 2018 y 2024 los procesos totales se multiplicaron por ocho, pero los
competitivos solo por 2,4. Lo que creció fue la contratación directa y el régimen
especial, donde no hay licitación a la cual presentarse.

**Para la empresa que licita, esa es una conclusión en sí misma:** el espacio donde
puede competir viene encogiéndose como proporción del total. Y solo se puede mostrar
teniendo los años viejos; con una ventana que arranque en 2020 se ve apenas el tramo
final de la caída.

De ahí la decisión de arrancar en 2017 y no en 2020:

| Tramo | Qué aporta |
|---|---|
| 2017–2019 | Unos 157.000 procesos competitivos, un 15 % más sobre el millón que hay de 2020 en adelante. Dos de ellos previos a la pandemia, que son la referencia de condiciones normales |
| 2015–2016 | Unos 6.000 procesos competitivos, y son los años más afectados por la adopción voluntaria de la plataforma |

Los años 2020 y 2021 están distorsionados por la emergencia sanitaria: contratación
directa por urgencia, plazos excepcionales. Si todo el histórico arrancara en 2020,
lo que el análisis tomaría como normalidad sería un periodo anormal.

**Las dos limitaciones hay que declararlas:**

Antes de 2020 publicar en SECOP II no era obligatorio para todas las entidades. Eso
sesga **qué entidades aparecen** en esos años, hacia las más organizadas. No
invalida sus procesos competitivos, pero sí hace que la mezcla de entidades no sea
directamente comparable entre tramos.

Y contamina parcialmente la tendencia de arriba: una parte de la caída en la
participación de lo competitivo puede ser contratación directa que llegó tarde a la
plataforma, no contratación directa que creció. Separar las dos cosas exigiría una
fuente externa que no está en este conjunto.

### 1.7 Por qué el corte cae un 1 de enero

Porque la contratación pública tiene un ritmo anual marcado. Los procesos publicados
en 2024, mes a mes:

| Mes | Procesos | |
|---|---:|---|
| Enero | 157.942 | |
| Febrero | 173.731 | ← el mes más activo |
| Marzo | 129.138 | |
| Abril | 130.709 | |
| Mayo | 138.756 | |
| Junio | 109.147 | |
| Julio | 124.633 | |
| Agosto | 147.795 | |
| Septiembre | 159.801 | |
| Octubre | 161.123 | |
| Noviembre | 130.769 | |
| Diciembre | 106.826 | ← el mes más flojo |

Entre el pico y el valle hay un 63 % de diferencia, y el patrón sigue la vigencia
fiscal, que en Colombia va de enero a diciembre: cuando entra el presupuesto nuevo
las entidades salen a contratar, y para diciembre ya gastaron.

De ahí salen dos consecuencias:

**Un corte a mitad de año deforma ese año.** Arrancar en julio de 2017 dejaría
afuera el pico de enero y febrero, y ese año aparecería como pequeño. Sería un
efecto del recorte, no una caída real de la contratación.

**Las comparaciones entre años requieren años completos.** Buena parte del análisis
consiste en seguir cómo se mueve una entidad de un año al siguiente, y un año
partido por la mitad no se puede comparar contra uno entero.

**Advertencia que hay que arrastrar al resto del cuaderno:** 2026 es un año
incompleto de todas formas, porque los datos llegan hasta septiembre. En cualquier
gráfico o comparación por año hay que señalarlo, o parecerá que la contratación se
desplomó en el último año.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 6  ·  Columnas que se le piden al portal
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Nada. Es una lista de nombres, no un cálculo.
# QUÉ HACE   : Deja escrito qué columnas se le piden al portal de cada base, y
#              cuáles se dejan afuera a propósito.
# QUÉ SALE   : Un conteo de cuántas se piden y cuántas se descartan.
# PARA QUÉ   : Esta lista es la que decide qué entra al equipo. Lo que no esté
#              aquí no se descarga nunca, y por eso los datos personales quedan
#              resueltos desde el principio y no después.
# ──────────────────────────────────────────────────────────────────────

COLUMNAS = {

    # ── PROCESOS · 35 de 60 ───────────────────────────────────────────
    "procesos": [
        # quién convoca
        "id_del_proceso", "nit_entidad", "entidad", "codigo_entidad",
        "departamento_entidad", "ciudad_entidad", "ordenentidad",
        # qué se contrata y cómo
        "modalidad_de_contratacion", "tipo_de_contrato", "subtipo_de_contrato",
        "codigo_principal_de_categoria", "fase", "duracion",
        "unidad_de_duracion", "precio_base", "numero_de_lotes",
        # cuándo
        "fecha_de_publicacion_del", "fecha_de_recepcion_de", "fecha_adjudicacion",
        # cuánta competencia hubo
        "proveedores_invitados", "proveedores_con_invitacion",
        "visualizaciones_del", "proveedores_que_manifestaron",
        "respuestas_al_procedimiento", "respuestas_externas",
        "conteo_de_respuestas_a_ofertas", "proveedores_unicos_con",
        # en qué terminó  ← de aquí sale la variable objetivo
        "adjudicado", "estado_del_procedimiento", "estado_resumen",
        "nit_del_proveedor_adjudicado", "codigoproveedor",
        "valor_total_adjudicacion", "departamento_proveedor", "ciudad_proveedor",
    ],

    # ── CONTRATOS · 38 de 92 ──────────────────────────────────────────
    "contratos": [
        # quién contrata y a quién
        "id_contrato", "proceso_de_compra", "nit_entidad", "nombre_entidad",
        "codigo_entidad", "documento_proveedor", "codigo_proveedor",
        "departamento", "ciudad", "orden", "sector", "rama",
        "entidad_centralizada", "es_grupo", "es_pyme",
        # qué contrato es
        "estado_contrato", "codigo_de_categoria_principal", "tipo_de_contrato",
        "modalidad_de_contratacion", "origen_de_los_recursos", "destino_gasto",
        "habilita_pago_adelantado", "liquidaci_n", "espostconflicto",
        # cuándo y por cuánto tiempo
        "fecha_de_firma", "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato", "duraci_n_del_contrato",
        "dias_adicionados", "ultima_actualizacion",
        # el dinero  ← de aquí salen las variables de control
        "valor_del_contrato", "valor_facturado", "valor_pagado",
        "valor_pendiente_de_pago", "valor_amortizado",
        "valor_pendiente_de_ejecucion", "saldo_cdp", "saldo_vigencia",
    ],

    # ── PROVEEDORES · 15 de 26 ────────────────────────────────────────
    "proveedores": [
        "codigo", "nit", "es_entidad", "es_grupo", "esta_activa",
        "fecha_creacion", "codigo_categoria_principal",
        "descripcion_categoria_principal", "pais", "departamento", "municipio",
        "tipo_empresa", "espyme", "ubicacion", "tama_o_entidad",
    ],

    # ── GRUPOS · 20 de 33 ─────────────────────────────────────────────
    "grupos": [
        "codigo_grupo", "nit_grupo", "es_entidad", "es_proveedor",
        "esta_activo", "fecha_creaci_n_grupo",
        "codigo_categor_a_principal_grupo", "pais_grupo", "departamento_grupo",
        "minucipio", "tipo_empresa_grupo", "es_mipyme", "ubicaci_n_grupo",
        "codigo_participante", "nit_participante", "tipo_empresa_participante",
        "ubicaci_n_participante", "fecha_creaci_n_participante",
        "participacion", "es_lider_del_grupo",
    ],
}


# Cuántas columnas tiene cada base en el portal, para reportar cuántas quedan
# afuera.
TOTAL_EN_PORTAL = {"procesos": 60, "contratos": 92, "proveedores": 26, "grupos": 33}


# ── Qué se dejó afuera y por qué ──────────────────────────────────────
# Esto no es código: es la justificación, escrita donde se toma la decisión.
#
#   DATOS PERSONALES
#     Contratos : nombre, cédula, domicilio y género del representante legal;
#                 banco, tipo y número de cuenta; nombre y documento de
#                 supervisores y ordenadores del gasto.
#     Proveedores y Grupos : teléfono, fax, correo, dirección, página web, y
#                 los mismos datos del representante legal.
#
#   NOMBRES DE PROVEEDOR
#     También queda afuera el nombre de las empresas proveedoras. Buena parte
#     de los proveedores del Estado son personas naturales, de modo que esa
#     columna contiene nombres de personas. Para identificar a un proveedor
#     basta el NIT, que es lo que se usa para cruzar las tablas. El nombre de
#     las entidades públicas sí se conserva: son instituciones.
#
#   TEXTO LIBRE LARGO
#     La descripción del procedimiento y el objeto del contrato. Pesan más que
#     todas las demás columnas juntas y no entran en ningún cálculo de esta
#     fase. Una clasificación por lo que se compra exigiría volver a pedirlas.
#
#   SIN VALOR PARA EL ANÁLISIS
#     Direcciones de páginas web, códigos internos del sistema, y fechas de
#     fases intermedias que vienen vacías en la gran mayoría de los registros.

print("COLUMNAS QUE SE LE PIDEN AL PORTAL")
print("-" * 74)
print(f"  {'base':14}{'se piden':>10}{'en el portal':>14}{'se dejan':>10}")
print("-" * 74)
for base, cols in COLUMNAS.items():
    total = TOTAL_EN_PORTAL[base]
    print(f"  {base:14}{len(cols):>10}{total:>14}{total-len(cols):>10}")
print("-" * 74)
pedidas = sum(len(c) for c in COLUMNAS.values())
totales = sum(TOTAL_EN_PORTAL.values())
print(f"  {'TOTAL':14}{pedidas:>10}{totales:>14}{totales-pedidas:>10}")
print(f"\n  Se descarta el {(totales-pedidas)/totales*100:.0f} % de las columnas.")
print(f"  Ninguna columna con datos personales entra al equipo.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 7  ·  Función de consulta y comprobación de columnas
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : El código de una base y la lista de columnas requeridas.
# QUÉ HACE   : Arma la dirección de consulta, se la pide al portal y convierte
#              la respuesta en una tabla. Si la conexión falla, espera y vuelve
#              a intentar hasta cuatro veces, con una pausa mayor cada vez.
# QUÉ SALE   : Una tabla de pandas con las filas pedidas.
# PARA QUÉ   : Es la pieza que usa todo el paso 1. Abajo se prueba pidiendo dos
#              filas de cada base, para confirmar que los 108 nombres de columna
#              existen. Un nombre mal escrito debe aparecer ahora y no a mitad
#              de una descarga de dos horas.
# ──────────────────────────────────────────────────────────────────────

import io


def pedir(codigo, columnas, donde=None, limite=50000, salto=0, intentos=4):
    """Pide un pedazo de una base al portal y lo devuelve como tabla.

    Las instrucciones que van en la consulta:
      $select  qué columnas
      $where   qué filas
      $limit   cuántas de una vez
      $offset  desde cuál empezar a contar
      $order   en qué orden, para que las tandas no se repitan ni se salten
               filas; ":id" es el identificador interno que el portal le asigna
               a cada registro, y es el orden más estable disponible.

    Todo se lee como texto a propósito. Dejar que el programa adivine los tipos
    haría que interpretara mal los números escritos a la colombiana y que
    convirtiera en vacío los textos que dicen "No Definido", que es justo lo que
    hay que poder contar más adelante.
    """
    url = f"{PORTAL}/{codigo}.csv"
    consulta = {
        "$select": ",".join(columnas),
        "$limit":  limite,
        "$offset": salto,
        "$order":  ":id",
    }
    if donde:
        consulta["$where"] = donde

    for intento in range(1, intentos + 1):
        try:
            r = requests.get(url, params=consulta, timeout=300)
            r.raise_for_status()
            return pd.read_csv(io.StringIO(r.text), dtype=str,
                               keep_default_na=False, na_values=[""])
        except Exception as e:
            if intento == intentos:
                raise
            espera = 5 * intento
            print(f"      falló el intento {intento} ({type(e).__name__}), "
                  f"reintento en {espera} s")
            time.sleep(espera)


# ── Prueba: dos filas de cada base ────────────────────────────────────
print("COMPROBANDO QUE LAS COLUMNAS EXISTAN")
print("=" * 74)

muestras = {}
for base, info in BASES.items():
    pedidas = COLUMNAS[base]
    print(f"\n{base.upper()}  ({info['codigo']})")
    try:
        m = pedir(info["codigo"], pedidas, limite=2)
        muestras[base] = m
        faltan  = [c for c in pedidas if c not in m.columns]
        sobran  = [c for c in m.columns if c not in pedidas]
        print(f"   pedidas {len(pedidas)}   ·   llegaron {len(m.columns)}   "
              f"·   filas de prueba {len(m)}")
        if faltan:
            print(f"   NO EXISTEN EN EL PORTAL: {faltan}")
        if sobran:
            print(f"   llegaron sin pedirlas: {sobran}")
        if not faltan and not sobran:
            print(f"   todas correctas")
    except Exception as e:
        print(f"   FALLÓ: {type(e).__name__}: {e}")

print("\n" + "=" * 74)
if len(muestras) == 4 and all(
        not [c for c in COLUMNAS[b] if c not in m.columns]
        for b, m in muestras.items()):
    print("  Las cuatro bases responden y los 108 nombres existen.")
    print("  Se puede lanzar la descarga.")
else:
    print("  Hay algo que corregir antes de descargar. Ver los avisos de arriba.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 8  ·  Funciones de descarga
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : El nombre de una base y, para las fechadas, un mes.
# QUÉ HACE   : Baja ese pedazo de base en tandas de cincuenta mil filas, las
#              junta y las guarda en un archivo propio. Un archivo que ya exista
#              no se vuelve a bajar, pero sí se cuentan sus filas para que los
#              resúmenes muestren los totales reales y no ceros.
# QUÉ SALE   : Archivos CSV dentro de la carpeta "datos": uno por mes en el caso
#              de procesos y contratos, uno por tanda en el de las listas.
# PARA QUÉ   : Que la descarga se pueda interrumpir y retomar. Si se corta en el
#              mes cuarenta, la siguiente corrida salta los treinta y nueve ya
#              guardados y continúa desde ahí.
# ──────────────────────────────────────────────────────────────────────

def meses_del_periodo(desde, hasta):
    """Devuelve la lista de meses entre dos fechas, como pares (año, mes)."""
    return [(p.year, p.month) for p in pd.period_range(desde, hasta, freq="M")]


def contar_filas(ruta, columna):
    """Cuenta las filas de un CSV ya descargado, sin cargarlo entero.

    Entra: la ruta de un archivo y el nombre de una columna cualquiera.
    Sale:  cuántas filas tiene.

    Se lee una sola columna en vez del archivo completo, que es muchas veces más
    rápido. El resultado queda guardado en un archivo de conteos al lado, de modo
    que la siguiente corrida lo lee de ahí y ni siquiera abre el CSV.

    No se cuentan los saltos de línea del archivo porque un campo de texto puede
    contener saltos adentro, y ese conteo daría de más.
    """
    registro = ruta.parent / "conteo.json"
    conteos = json.loads(registro.read_text()) if registro.exists() else {}

    if ruta.name in conteos:
        return conteos[ruta.name]

    n = len(pd.read_csv(ruta, usecols=[columna], dtype=str,
                        keep_default_na=False, na_values=[""]))
    conteos[ruta.name] = n
    registro.write_text(json.dumps(conteos, indent=1))
    return n


def bajar_todas_las_tandas(codigo, columnas, donde=None, limite=50000):
    """Pide tandas seguidas hasta que el portal deja de mandar filas.

    La señal de que ya no hay más es que una tanda venga con menos filas de las
    pedidas: significa que se agotó lo que había.
    """
    partes, salto = [], 0
    while True:
        tanda = pedir(codigo, columnas, donde=donde, limite=limite, salto=salto)
        if len(tanda) == 0:
            break
        partes.append(tanda)
        if len(tanda) < limite:
            break
        salto += limite
    return partes


def bajar_mes(base, anio, mes, limite=50000):
    """Baja un mes de una base fechada y lo guarda en su propio archivo."""
    info, columnas = BASES[base], COLUMNAS[base]
    carpeta = DATOS / base
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / f"{base}_{anio}-{mes:02d}.csv"

    if destino.exists():
        return "ya estaba", contar_filas(destino, columnas[0])

    # El rango va del primer día de este mes al primer día del siguiente, sin
    # incluirlo. Así ningún registro queda contado dos veces ni se escapa.
    desde = pd.Timestamp(anio, mes, 1)
    hasta = min(desde + pd.offsets.MonthBegin(1), CORTE + pd.Timedelta(days=1))
    if desde > CORTE:
        return "fuera del corte", 0

    campo = info["fecha"]
    donde = (f"{campo} >= '{desde:%Y-%m-%d}' AND "
             f"{campo} < '{hasta:%Y-%m-%d}'")

    partes = bajar_todas_las_tandas(info["codigo"], columnas, donde, limite)

    # Se escribe primero en un archivo temporal y solo al final se le pone el
    # nombre definitivo. Una caída a mitad de la escritura deja un archivo
    # ".parcial" que la siguiente corrida ignora, en vez de un CSV incompleto que
    # parecería estar bien.
    temporal = carpeta / f"{base}_{anio}-{mes:02d}.parcial"
    if partes:
        tabla = pd.concat(partes, ignore_index=True)
        tabla.to_csv(temporal, index=False, encoding="utf-8")
        filas = len(tabla)
    else:
        # Mes sin registros: queda el archivo con solo los encabezados, para que
        # la siguiente corrida sepa que ya se revisó y no lo vuelva a pedir.
        temporal.write_text(",".join(columnas) + "\n", encoding="utf-8")
        filas = 0

    temporal.replace(destino)
    return "descargado", filas


def bajar_catalogo(base, limite=50000):
    """Baja completa una de las listas, en tandas, una por archivo.

    Proveedores y Grupos no se parten por mes porque su fecha es la de
    inscripción y no sirve para repartir la descarga. Se bajan seguidas, y cada
    tanda queda en su propio archivo para poder retomar.
    """
    info, columnas = BASES[base], COLUMNAS[base]
    carpeta = DATOS / base
    carpeta.mkdir(parents=True, exist_ok=True)

    numero, salto, total = 0, 0, 0
    while True:
        destino = carpeta / f"{base}_parte_{numero:03d}.csv"
        if destino.exists():
            total += contar_filas(destino, columnas[0])
            numero += 1
            salto += limite
            continue

        tanda = pedir(info["codigo"], columnas, limite=limite, salto=salto)
        if len(tanda) == 0:
            break

        temporal = carpeta / f"{base}_parte_{numero:03d}.parcial"
        tanda.to_csv(temporal, index=False, encoding="utf-8")
        temporal.replace(destino)

        total += len(tanda)
        print(f"      parte {numero:03d}: {len(tanda):>6,} filas   "
              f"acumulado {total:>9,}".replace(",", "."))

        if len(tanda) < limite:
            break
        numero += 1
        salto += limite

    return total


print("Funciones de descarga listas:")
print("  meses_del_periodo(desde, hasta)   los meses entre dos fechas")
print("  contar_filas(ruta, columna)       filas de un archivo ya descargado")
print("  bajar_mes(base, año, mes)         un mes de procesos o contratos")
print("  bajar_catalogo(base)              una lista completa")
print("\nTodavía no se ha descargado nada. Eso ocurre en las celdas siguientes.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 9  ·  Descarga de las cuatro bases
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Nada local. Todo se le pide al portal.
# QUÉ HACE   : Primero baja un solo mes para medir el ritmo. Si el ritmo sirve,
#              baja las cuatro bases completas: Proveedores y Grupos en tandas,
#              Procesos y Contratos mes por mes.
# QUÉ SALE   : Los archivos dentro de datos/<base>, y un informe de cuántas
#              filas, cuántos archivos y cuántos minutos tomó cada una.
# PARA QUÉ   : Es la etapa de extracción del ETL. Antes de lanzar una descarga
#              de horas conviene conocer el ritmo: a treinta segundos por mes
#              son unos cuarenta minutos; a cinco minutos por mes habría que
#              replantear el alcance antes de empezar y no a mitad de camino.
#              Una interrupción no obliga a empezar de nuevo: los meses ya
#              bajados se saltan sin pedir nada.
# ──────────────────────────────────────────────────────────────────────

# ══ 1 · PRUEBA DE RITMO ═════════════════════════════════════════════════
print("PRUEBA DE DESCARGA · Procesos, marzo de 2024")
print("=" * 74)

reloj = time.perf_counter()
estado, filas = bajar_mes("procesos", 2024, 3)
tardo = time.perf_counter() - reloj

archivo = DATOS / "procesos" / "procesos_2024-03.csv"
peso_mb = archivo.stat().st_size / 1024**2 if archivo.exists() else 0

print(f"  Estado            : {estado}")
print(f"  Filas             : {filas:,}".replace(",", "."))
print(f"  Tiempo            : {tardo:.1f} segundos")
print(f"  Peso del archivo  : {peso_mb:.1f} MB")
if filas:
    print(f"  Ritmo             : {filas/tardo:,.0f} filas por segundo"
          .replace(",", "."))

meses = meses_del_periodo(INICIO, CORTE)
print(f"\nPROYECCIÓN")
print("-" * 74)
print(f"  Meses por bajar, por base : {len(meses)}  "
      f"(de {INICIO:%Y-%m} a {CORTE:%Y-%m})")
if filas and tardo:
    # marzo de 2024 tuvo 129.138 procesos; sirve como mes de referencia para
    # estimar el resto, sabiendo que unos meses son más pesados que otros
    print(f"  Procesos, estimado        : {len(meses)*tardo/60:.0f} minutos   "
          f"·   {len(meses)*peso_mb/1024:.1f} GB")
    print(f"  Contratos, estimado       : similar, posiblemente algo menos")
    print(f"\n  Son estimaciones gruesas: dependen de la conexión y de la carga")
    print(f"  del portal en ese momento.")


# ══ 2 · LAS DOS LISTAS COMPLETAS ════════════════════════════════════════
# Proveedores y Grupos van primero porque son las más rápidas: un error de
# armado se descubre en minutos y no en una hora.
print("\n" + "=" * 74)
print("DESCARGA DE LAS DOS LISTAS")
print("=" * 74)

resumen = {}
for base in ("proveedores", "grupos"):
    print(f"\n{base.upper()}  ({BASES[base]['codigo']})")
    reloj = time.perf_counter()
    filas = bajar_catalogo(base)
    tardo = time.perf_counter() - reloj

    carpeta = DATOS / base
    archivos = sorted(carpeta.glob(f"{base}_parte_*.csv"))
    peso = sum(a.stat().st_size for a in archivos) / 1024**2

    resumen[base] = {"filas": filas, "archivos": len(archivos),
                     "mb": peso, "segundos": tardo}
    print(f"   terminado: {filas:,} filas en {len(archivos)} archivos   "
          .replace(",", ".") + f"{peso:.0f} MB   {tardo/60:.1f} min")


# ══ 3 · LAS DOS BASES GRANDES, MES POR MES ══════════════════════════════
print("\n" + "=" * 78)
print(f"DESCARGA COMPLETA · {len(meses)} meses por base")
print(f"De {INICIO:%B de %Y} a {CORTE:%B de %Y}")
print("=" * 78)

informe = {}
for base in ("procesos", "contratos"):
    print(f"\n{base.upper()}  ({BASES[base]['codigo']})")
    print("-" * 78)

    reloj = time.perf_counter()
    filas_base, bajados, saltados = 0, 0, 0

    for i, (anio, mes) in enumerate(meses, 1):
        estado, filas = bajar_mes(base, anio, mes)
        filas_base += filas

        if estado == "ya estaba":
            saltados += 1
            continue

        bajados += 1
        transcurrido = time.perf_counter() - reloj
        # lo que falta se estima con el ritmo de lo que lleva bajado
        faltan = (len(meses) - i) * (transcurrido / max(bajados, 1))
        print(f"  {anio}-{mes:02d}   {filas:>8,} filas".replace(",", ".")
              + f"   acumulado {filas_base:>10,}".replace(",", ".")
              + f"   faltan ~{faltan/60:>4.0f} min")

    tardo = time.perf_counter() - reloj
    carpeta = DATOS / base
    archivos = sorted(carpeta.glob(f"{base}_*.csv"))
    peso = sum(a.stat().st_size for a in archivos) / 1024**3

    informe[base] = {"filas": filas_base, "archivos": len(archivos),
                     "gb": peso, "minutos": tardo / 60,
                     "bajados": bajados, "saltados": saltados}

    print(f"\n  {base}: {filas_base:,} filas".replace(",", ".")
          + f"   {len(archivos)} archivos   {peso:.2f} GB   {tardo/60:.0f} min")
    if saltados:
        print(f"  ({saltados} meses ya estaban de una corrida anterior)")


# ══ 4 · RESUMEN DE LA EXTRACCIÓN ════════════════════════════════════════
print("\n" + "=" * 78)
print("DESCARGA TERMINADA")
print("-" * 78)
print(f"  {'base':14}{'filas':>14}{'archivos':>10}{'tamaño':>10}{'minutos':>10}")
print("-" * 78)
for base, r in resumen.items():
    print(f"  {base:14}{r['filas']:>14,}".replace(",", ".")
          + f"{r['archivos']:>10}{r['mb']/1024:>9.2f} GB{r['segundos']/60:>10.1f}")
for base, r in informe.items():
    print(f"  {base:14}{r['filas']:>14,}".replace(",", ".")
          + f"{r['archivos']:>10}{r['gb']:>9.2f} GB{r['minutos']:>10.0f}")


`CELDA 10`

## Paso 2 · De muchos archivos a una tabla

### 2.1 Lo que quedó en el disco

La descarga dejó archivos repartidos en cuatro carpetas. Eso está bien para
bajar, porque permite retomar si algo se cae, pero no sirve para analizar: no se
puede calcular un promedio sobre 81 archivos sueltos.

El paso siguiente es juntarlos. Y aquí aparece el primer problema real de trabajar
con datos grandes.

### 2.2 Por qué no basta con abrirlos y pegarlos

Un archivo de texto ocupa una cosa en el disco y otra muy distinta en la memoria.
Al leer un CSV, cada valor deja de ser una tira de caracteres y pasa a ser un objeto
con su propia estructura. Para texto, esa estructura pesa varias veces lo que pesaba
el texto original.

Con 2,94 GB de archivos de Procesos, leerlos tal cual puede ocupar entre 10 y 15 GB
de memoria. En un equipo de 16 GB eso agota la memoria y el programa se cierra sin
explicación.

La salida no es leer menos datos: es **leerlos y convertirlos de inmediato**, antes
de acumularlos. Tres conversiones hacen casi todo el trabajo:

| Qué se convierte | En qué | Cuánto ahorra |
|---|---|---|
| Columnas de texto que se repiten mucho — modalidad, departamento, tipo de contrato | En etiquetas con un diccionario detrás | Enorme: en vez de guardar "Mínima cuantía" ocho millones de veces, se guarda una vez y ocho millones de referencias a ella |
| Columnas de números guardadas como texto | En números de verdad | Entre tres y cinco veces menos |
| Columnas de fecha guardadas como texto | En fechas de verdad | Unas cuatro veces menos |

Con eso, los 10 o 15 GB se reducen a dos o tres, y ahí sí caben cómodos.

### 2.3 El orden importa

Es tentador leer todo primero y convertir después. No funciona: para convertir hay
que tener los datos en memoria, y el problema era precisamente que no caben.

El orden correcto es archivo por archivo:

```
leer un mes  →  convertir  →  guardar el resultado  →  soltar el original
leer el siguiente mes  →  convertir  →  guardar  →  soltar
...
al final: unir todos los convertidos
```

Así nunca hay más de un mes sin convertir en memoria. Es más lento que leer todo de
golpe, pero es la diferencia entre que funcione y que no.

### 2.4 El formato en que se guarda lo convertido

Los archivos convertidos no se vuelven a guardar como CSV sino como **parquet**, que
es un formato pensado para tablas grandes. Tres diferencias que importan:

- **Guarda los tipos adentro.** Al volver a abrir el archivo, las fechas siguen
  siendo fechas y las etiquetas siguen siendo etiquetas, sin repetir la conversión.
- **Comprime por columna.** Como todos los valores de una columna son del mismo
  tipo, se comprimen mucho mejor que un CSV, que mezcla todo en cada renglón.
- **Permite leer solo algunas columnas.** No hay que cargar las 35 para consultar
  dos.

### 2.5 Lo que se arregla de una vez, mientras se lee

Hay cuatro defectos de la fuente que conviene corregir en este mismo paso, porque
hacerlo después significaría recorrer ocho millones de filas otra vez.

**Los textos que dicen "vacío".** La fuente escribe `No Definido` en vez de dejar la
celda en blanco. Al convertirlos a vacío de verdad, el porcentaje de datos faltantes
**sube**. Que suba es correcto: antes estaba escondido.

**Los números a la colombiana.** Acá el punto separa los miles y la coma los
decimales, al revés de lo que espera el programa. `15.693.000` se lee como quince
millones en Colombia y como quince punto seis afuera. Este es el error más costoso
de los cuatro porque no avisa: deja números plausibles pero mil veces más pequeños.

**Las dos formas de escribir una fecha.** Unas columnas vienen como
`2024-03-15T00:00:00.000` y otras en formato largo. Un solo intento de conversión
deja la mitad vacía.

**Las fechas imposibles.** Aparecen valores como `0001-01-01`, que no es una fecha
real sino la forma que tiene el sistema de decir que no hay dato.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 11  ·  Conversiones y mapa de tipos
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Columnas de texto tal como las entregó el portal.
# QUÉ HACE   : Define las conversiones que arreglan los defectos de la fuente, y
#              la tabla que dice de qué tipo es cada columna.
# QUÉ SALE   : Cuatro funciones listas y un mapa de tipos por base.
# PARA QUÉ   : Son las piezas que usa el lector de la celda siguiente. Van aparte
#              para poder probarlas solas: una conversión mal hecha se descubre
#              acá y no después de veinte minutos de lectura.
# ──────────────────────────────────────────────────────────────────────

def limpiar_texto(s):
    """Quita espacios sobrantes y convierte en vacío real los textos que
    significan vacío.

    Entra: una columna de texto con cosas como "  No Definido  ".
    Sale:  la misma columna, con esos valores convertidos en vacío de verdad.

    La comparación se hace en minúscula porque la fuente escribe el mismo texto
    de varias formas y así las tres caen en el mismo saco.
    """
    s = s.astype("string").str.strip()
    return s.where(~s.str.lower().isin(VACIOS))


def a_numero(s):
    """Convierte texto a número respetando cómo se escriben acá las cifras.

    Entra: "1.234.567,89" o "15.693.000" o "1234,5"
    Sale:  1234567.89     o  15693000.0   o  1234.5

    Hay tres casos y cada uno se resuelve distinto, porque en Colombia el punto
    separa los miles y la coma los decimales, al revés de lo que espera el
    programa:

      con punto y coma   ->  el punto es de miles, la coma es decimal
      solo puntos        ->  si el último grupo tiene 3 dígitos, son miles
      solo coma          ->  si el último grupo tiene 3 dígitos son miles,
                             si no, es decimal

    Un reemplazo ingenuo de caracteres convertiría 15.693.000 en 15,693: el error
    más caro de todos, porque no avisa y deja números plausibles.
    """
    s = s.astype("string").str.strip()
    s = s.str.replace(r"[^\d,.\-]", "", regex=True)

    hay_punto = s.str.contains(r"\.", na=False)
    hay_coma  = s.str.contains(",", na=False)

    # caso 1: los dos separadores
    s = s.mask(hay_punto & hay_coma,
               s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

    # caso 2: solo puntos, y el último grupo tiene 3 dígitos
    ultimo3 = s.str.extract(r"\.(\d+)$", expand=False).str.len().eq(3).fillna(False)
    s = s.mask(hay_punto & ~hay_coma & ultimo3, s.str.replace(".", "", regex=False))

    # caso 3: solo coma
    ultimo3c = s.str.extract(r",(\d+)$", expand=False).str.len().eq(3).fillna(False)
    s = s.mask(hay_coma & ~hay_punto & ultimo3c,  s.str.replace(",", "", regex=False))
    s = s.mask(hay_coma & ~hay_punto & ~ultimo3c, s.str.replace(",", ".", regex=False))

    return pd.to_numeric(s, errors="coerce")


def a_fecha(s):
    """Convierte texto a fecha y descarta las fechas imposibles.

    Entra: "2024-03-15T00:00:00.000" o un texto vacío o "0001-01-01"
    Sale:  una fecha de verdad, o vacío

    Lo anterior a 2010 o posterior al corte no se descarta por ser raro: se
    descarta porque no puede existir. SECOP II arrancó en 2015 y los datos llegan
    hasta septiembre de 2026, así que una fecha fuera de ese rango es un error de
    captura o la forma que tiene el sistema de decir "no hay dato".
    """
    f = pd.to_datetime(s, errors="coerce", format="ISO8601")
    imposible = (f < pd.Timestamp("2010-01-01")) | (f > CORTE + pd.Timedelta(days=400))
    return f.mask(imposible)


def a_etiqueta(s):
    """Convierte una columna de texto repetido en etiquetas.

    Entra: ocho millones de veces el texto "Mínima cuantía"
    Sale:  el texto guardado una sola vez, y ocho millones de referencias a él

    Es la conversión que más memoria ahorra. Solo sirve cuando los valores se
    repiten mucho; en una columna de identificadores únicos haría lo contrario.
    """
    return limpiar_texto(s).astype("category")


# ── De qué tipo es cada columna ───────────────────────────────────────
# "llave" son identificadores que se usan para cruzar tablas: se dejan como texto
# porque no se opera con ellos y porque un NIT con ceros adelante se dañaría al
# convertirlo en número.

TIPOS = {

    "procesos": {
        "llave": ["id_del_proceso", "nit_entidad", "codigo_entidad",
                  "nit_del_proveedor_adjudicado", "codigoproveedor"],
        "etiqueta": ["entidad", "departamento_entidad", "ciudad_entidad",
                     "ordenentidad", "modalidad_de_contratacion",
                     "tipo_de_contrato", "subtipo_de_contrato",
                     "codigo_principal_de_categoria", "fase",
                     "unidad_de_duracion", "adjudicado",
                     "estado_del_procedimiento", "estado_resumen",
                     "departamento_proveedor", "ciudad_proveedor"],
        "numero": ["duracion", "precio_base", "numero_de_lotes",
                   "proveedores_invitados", "proveedores_con_invitacion",
                   "visualizaciones_del", "proveedores_que_manifestaron",
                   "respuestas_al_procedimiento", "respuestas_externas",
                   "conteo_de_respuestas_a_ofertas", "proveedores_unicos_con",
                   "valor_total_adjudicacion"],
        "fecha": ["fecha_de_publicacion_del", "fecha_de_recepcion_de",
                  "fecha_adjudicacion"],
    },

    "contratos": {
        # duraci_n_del_contrato queda como texto a propósito: viene escrita como
        # "69 Dia(s)" o "3 Mes(es)", así que el número solo no significa nada sin
        # la unidad. Se resuelve más adelante, al construir las variables.
        "llave": ["id_contrato", "proceso_de_compra", "nit_entidad",
                  "codigo_entidad", "documento_proveedor", "codigo_proveedor",
                  "duraci_n_del_contrato"],
        "etiqueta": ["nombre_entidad", "departamento", "ciudad", "orden",
                     "sector", "rama", "entidad_centralizada", "es_grupo",
                     "es_pyme", "estado_contrato",
                     "codigo_de_categoria_principal", "tipo_de_contrato",
                     "modalidad_de_contratacion", "origen_de_los_recursos",
                     "destino_gasto", "habilita_pago_adelantado",
                     "liquidaci_n", "espostconflicto"],
        "numero": ["dias_adicionados", "valor_del_contrato", "valor_facturado",
                   "valor_pagado", "valor_pendiente_de_pago",
                   "valor_amortizado", "valor_pendiente_de_ejecucion",
                   "saldo_cdp", "saldo_vigencia"],
        "fecha": ["fecha_de_firma", "fecha_de_inicio_del_contrato",
                  "fecha_de_fin_del_contrato", "ultima_actualizacion"],
    },

    "proveedores": {
        "llave": ["codigo", "nit"],
        "etiqueta": ["es_entidad", "es_grupo", "esta_activa",
                     "codigo_categoria_principal",
                     "descripcion_categoria_principal", "pais", "departamento",
                     "municipio", "tipo_empresa", "espyme", "ubicacion",
                     "tama_o_entidad"],
        "numero": [],
        "fecha": ["fecha_creacion"],
    },

    "grupos": {
        "llave": ["codigo_grupo", "nit_grupo", "codigo_participante",
                  "nit_participante"],
        "etiqueta": ["es_entidad", "es_proveedor", "esta_activo",
                     "codigo_categor_a_principal_grupo", "pais_grupo",
                     "departamento_grupo", "minucipio", "tipo_empresa_grupo",
                     "es_mipyme", "ubicaci_n_grupo",
                     "tipo_empresa_participante", "ubicaci_n_participante",
                     "es_lider_del_grupo"],
        "numero": ["participacion"],
        "fecha": ["fecha_creaci_n_grupo", "fecha_creaci_n_participante"],
    },
}


# ── Revisión de que el mapa esté completo ─────────────────────────────
print("REVISIÓN DEL MAPA DE TIPOS")
print("-" * 74)
print(f"  {'base':14}{'llave':>8}{'etiqueta':>10}{'numero':>8}{'fecha':>7}"
      f"{'suma':>7}{'pedidas':>9}")
print("-" * 74)
todo_bien = True
for base, tipos in TIPOS.items():
    suma = sum(len(v) for v in tipos.values())
    pedidas = len(COLUMNAS[base])
    marca = "" if suma == pedidas else "   <-- NO CUADRA"
    if suma != pedidas:
        todo_bien = False
        faltan = set(COLUMNAS[base]) - set(sum(tipos.values(), []))
        sobran = set(sum(tipos.values(), [])) - set(COLUMNAS[base])
        marca += f"\n      sin clasificar: {faltan}\n      de más: {sobran}"
    print(f"  {base:14}{len(tipos['llave']):>8}{len(tipos['etiqueta']):>10}"
          f"{len(tipos['numero']):>8}{len(tipos['fecha']):>7}"
          f"{suma:>7}{pedidas:>9}{marca}")

print("-" * 74)
print("  Todas las columnas están clasificadas." if todo_bien
      else "  Hay columnas sin clasificar. Revisar arriba.")

# ── Prueba rápida de las conversiones ─────────────────────────────────
print("\nPRUEBA DE LAS CONVERSIONES")
print("-" * 74)
ejemplo = pd.Series(["1.234.567,89", "15.693.000", "1234,5", "No Definido", ""])
print("  texto -> número")
for t, n in zip(ejemplo, a_numero(ejemplo)):
    print(f"    {str(t):16} -> {n}")

fechas = pd.Series(["2024-03-15T00:00:00.000", "0001-01-01T00:00:00.000",
                    "2019-11-09T00:00:00.000", ""])
print("  texto -> fecha")
for t, f in zip(fechas, a_fecha(fechas)):
    print(f"    {str(t):26} -> {f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELDA 12  ·  Conversión de los archivos a parquet
# ══════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Los archivos CSV descargados.
# QUÉ HACE   : Lee cada uno, le aplica las conversiones y lo vuelve a guardar en
#              parquet. Trabaja de a un archivo: nunca hay más de un mes sin
#              convertir en memoria.
# QUÉ SALE   : Una carpeta "convertido" dentro de cada base, con un archivo por
#              cada CSV original, y un informe de cuánto se redujo.
# PARA QUÉ   : Es el paso que hace que los datos quepan. El parquet guarda los
#              tipos dentro del archivo, así que al volver a abrirlos no hay que
#              repetir ninguna conversión.
# ──────────────────────────────────────────────────────────────────────

import psutil

ram = psutil.virtual_memory()
print(f"Memoria del equipo : {ram.total/1024**3:.1f} GB   "
      f"(libre ahora: {ram.available/1024**3:.1f} GB)")
print(f"Núcleos            : {psutil.cpu_count(logical=False)} físicos, "
      f"{psutil.cpu_count()} lógicos\n")


def ajustar_numero(s):
    """Guarda cada número en el tamaño más pequeño donde quepa sin dañarse.

    Entra: una columna de números recién convertida.
    Sale:  la misma columna, ocupando menos.

    Los conteos (cuántos oferentes, cuántos lotes) son enteros pequeños y caben
    en un formato de cuatro bytes. Las cifras de dinero NO se tocan: llegan a los
    billones, y achicarlas les quitaría precisión en los dígitos finales sin
    avisar.
    """
    if s.notna().any() and s.max() < 2_000_000 and (s.dropna() % 1 == 0).all():
        return s.astype("Int32")
    return s


def convertir_archivo(ruta, base):
    """Lee un CSV, le aplica las conversiones y devuelve la tabla lista."""
    tipos = TIPOS[base]
    t = pd.read_csv(ruta, dtype=str, keep_default_na=False,
                    na_values=[""], low_memory=False)

    for c in tipos["llave"]:
        if c in t:
            t[c] = limpiar_texto(t[c])
    for c in tipos["etiqueta"]:
        if c in t:
            t[c] = a_etiqueta(t[c])
    for c in tipos["numero"]:
        if c in t:
            t[c] = ajustar_numero(a_numero(t[c]))
    for c in tipos["fecha"]:
        if c in t:
            t[c] = a_fecha(t[c])
    return t


def convertir_base(base):
    """Convierte todos los archivos de una base, uno por uno."""
    origen = DATOS / base
    destino = origen / "convertido"
    destino.mkdir(exist_ok=True)

    archivos = sorted(origen.glob(f"{base}_*.csv"))
    csv_mb, pq_mb, filas, hechos, saltados = 0, 0, 0, 0, 0
    reloj = time.perf_counter()

    for i, ruta in enumerate(archivos, 1):
        salida = destino / (ruta.stem + ".parquet")
        csv_mb += ruta.stat().st_size / 1024**2

        if salida.exists():
            pq_mb += salida.stat().st_size / 1024**2
            filas += pd.read_parquet(salida, columns=[TIPOS[base]["llave"][0]]).shape[0]
            saltados += 1
            continue

        t = convertir_archivo(ruta, base)
        t.to_parquet(salida, index=False, compression="snappy")

        filas += len(t)
        pq_mb += salida.stat().st_size / 1024**2
        hechos += 1
        del t
        gc.collect()

        if i % 10 == 0 or i == len(archivos):
            print(f"      {i:>3}/{len(archivos)} archivos   "
                  f"{filas:>10,} filas".replace(",", ".")
                  + f"   {time.perf_counter()-reloj:>5.0f} s")

    return {"archivos": len(archivos), "filas": filas, "csv_mb": csv_mb,
            "pq_mb": pq_mb, "segundos": time.perf_counter() - reloj,
            "convertidos": hechos, "ya_estaban": saltados}


print("CONVIRTIENDO LAS CUATRO BASES")
print("=" * 78)

conversion = {}
for base in ("proveedores", "grupos", "contratos", "procesos"):
    print(f"\n{base.upper()}")
    conversion[base] = convertir_base(base)
    r = conversion[base]
    print(f"   {r['filas']:,} filas".replace(",", ".")
          + f"   ·   {r['csv_mb']:.0f} MB en CSV  ->  {r['pq_mb']:.0f} MB convertido"
          + f"   ·   {r['segundos']/60:.1f} min")

print("\n" + "=" * 78)
print("RESUMEN DE LA CONVERSIÓN")
print("-" * 78)
print(f"  {'base':14}{'filas':>12}{'CSV MB':>10}{'convertido MB':>16}{'reduce a':>11}")
print("-" * 78)
tot_csv = tot_pq = 0
for base, r in conversion.items():
    tot_csv += r["csv_mb"]; tot_pq += r["pq_mb"]
    print(f"  {base:14}{r['filas']:>12,}".replace(",", ".")
          + f"{r['csv_mb']:>10.0f}{r['pq_mb']:>16.0f}"
          + f"{r['pq_mb']/max(r['csv_mb'],1)*100:>10.0f} %")
print("-" * 78)
print(f"  {'TOTAL':14}{'':>12}{tot_csv:>10.0f}{tot_pq:>16.0f}"
      f"{tot_pq/max(tot_csv,1)*100:>10.0f} %")

`CELDA 13`

## Paso 3 · Cuánto cabe en memoria

### 3.1 Por qué este paso existe

Los datos convertidos pesan 645 MB en el disco. El equipo tiene 7,9 GB de memoria.
Parece que sobra espacio, pero esa comparación es engañosa: **un archivo ocupa más
en memoria que en el disco**, y en el caso de parquet mucho más, porque en el disco
está comprimido y en memoria no.

Un archivo de 234 MB puede ocupar entre uno y dos gigabytes al abrirse. Con Windows,
VS Code y el navegador compitiendo por la misma memoria, eso deja poco margen.

Hay dos formas de equivocarse aquí, y las dos cuestan tiempo:

**Asumir que cabe y lanzarlo.** Si no cabe, el programa no avisa: la memoria se
llena, Windows empieza a mover datos al disco para compensar, todo se vuelve
lentísimo y al final el kernel se reinicia. Media hora perdida sin un mensaje de
error que explique qué pasó.

**Asumir que no cabe y trabajar por pedazos sin necesidad.** Procesar por partes
complica el código y obliga a resolver problemas que no existirían si todo cupiera.

La salida es medir. Esta celda abre un mes, mide cuánto ocupó, y multiplica.

### 3.2 Qué se hace con el resultado

El conjunto completo de Procesos son 8,6 millones de filas. Según lo que mida la
celda siguiente, hay dos caminos:

| Si la proyección da | Entonces |
|---|---|
| Menos de 2 GB | Se carga completo y se trabaja normal |
| Más de 2 GB | Se trabaja **por año**: se arma la tabla de cada año por separado y se guarda, sin tener nunca todos los años al tiempo |

El segundo camino no pierde ningún dato: reparte el trabajo en el tiempo en vez de
exigirlo todo de golpe. Tiene además una ventaja para la fase siguiente, que va a
correr en los equipos del resto del equipo: una tabla guardada por año se puede leer
por año, así que no obliga a nadie a tener 2 GB libres para empezar.

### 3.3 La otra medición que sale de aquí

De paso se mide **cuánto ocupa cada columna**. Ese dato sirve para algo concreto:
casi siempre hay dos o tres columnas que se llevan la mitad de la memoria, y son las
de texto con valores únicos, como los identificadores. Saber cuáles son permite
decidir si alguna se puede dejar afuera de la tabla final.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 14  ·  Medición de memoria y decisión de estrategia
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Un mes de Procesos y uno de Contratos, ya convertidos.
# QUÉ HACE   : Los abre, mide cuánta memoria ocuparon y proyecta cuánto
#              ocuparía la base completa. También mide columna por columna,
#              para ver cuáles pesan más.
# QUÉ SALE   : La proyección de cada base y la decisión de si se puede
#              trabajar con todo cargado o hay que repartir por año.
# PARA QUÉ   : Para no descubrir a mitad del trabajo que no cabe. Un kernel
#              que se reinicia por falta de memoria no deja mensaje de error:
#              solo se muere, y hay que adivinar por qué.
# ──────────────────────────────────────────────────────────────────────

LIMITE_GB = 2.0   # por encima de esto se trabaja por año


def medir(base, archivo_muestra=None):
    """Abre un archivo convertido y proyecta el tamaño de la base completa.

    Entra: el nombre de una base.
    Sale:  un diccionario con lo que ocupa la muestra, cuánto se proyecta para
           el total, y el peso de cada columna.

    La proyección es una regla de tres: si un mes con cien mil filas ocupa
    treinta megas, ocho millones de filas ocupan ochenta veces eso. Es
    aproximada porque unos meses tienen textos más largos que otros, pero
    sirve para decidir.
    """
    carpeta = DATOS / base / "convertido"
    archivos = sorted(carpeta.glob("*.parquet"))
    if not archivos:
        return None

    # se toma el archivo más grande, que es el caso peor
    muestra = archivo_muestra or max(archivos, key=lambda a: a.stat().st_size)
    t = pd.read_parquet(muestra)

    # memory_usage(deep=True) mide también lo que ocupan los textos por dentro,
    # no solo la referencia a ellos. Sin "deep" la cifra sale muy por debajo.
    por_columna = t.memory_usage(deep=True).drop("Index", errors="ignore")
    mb_muestra = por_columna.sum() / 1024**2

    filas_total = conversion[base]["filas"]
    proyeccion_gb = mb_muestra / len(t) * filas_total / 1024

    del t
    gc.collect()

    return {
        "archivo": muestra.name,
        "filas_muestra": len(archivos) and len(pd.read_parquet(muestra,
                         columns=[TIPOS[base]["llave"][0]])),
        "mb_muestra": mb_muestra,
        "filas_total": filas_total,
        "proyeccion_gb": proyeccion_gb,
        "por_columna": (por_columna / 1024**2).sort_values(ascending=False),
    }


ram = psutil.virtual_memory()
print(f"Memoria del equipo : {ram.total/1024**3:.1f} GB")
print(f"Libre en este momento : {ram.available/1024**3:.1f} GB")
print(f"Umbral para decidir : {LIMITE_GB} GB por base\n")

print("PROYECCIÓN DE MEMORIA POR BASE")
print("=" * 78)
print(f"  {'base':14}{'filas':>12}{'muestra MB':>13}{'proyección GB':>16}{'veredicto':>14}")
print("-" * 78)

medidas, por_anio = {}, []
for base in ("proveedores", "grupos", "contratos", "procesos"):
    m = medir(base)
    if m is None:
        print(f"  {base:14}  sin archivos convertidos")
        continue
    medidas[base] = m
    cabe = m["proyeccion_gb"] <= LIMITE_GB
    if not cabe:
        por_anio.append(base)
    print(f"  {base:14}{m['filas_total']:>12,}".replace(",", ".")
          + f"{m['mb_muestra']:>13.0f}{m['proyeccion_gb']:>16.2f}"
          + f"{'cabe' if cabe else 'POR AÑO':>14}")

total_gb = sum(m["proyeccion_gb"] for m in medidas.values())
print("-" * 78)
print(f"  {'TODAS JUNTAS':14}{'':>12}{'':>13}{total_gb:>16.2f}")

# ── Las columnas que más pesan ────────────────────────────────────────
print(f"\nLAS SEIS COLUMNAS MÁS PESADAS DE CADA BASE GRANDE")
print("=" * 78)
for base in ("procesos", "contratos"):
    if base not in medidas:
        continue
    m = medidas[base]
    print(f"\n  {base.upper()}   (muestra: {m['archivo']})")
    for col, mb in m["por_columna"].head(6).items():
        parte = mb / m["mb_muestra"] * 100
        barra = "█" * int(parte / 2)
        print(f"    {col:34} {mb:>7.1f} MB  {parte:>5.1f} %  {barra}")

# ── Decisión ──────────────────────────────────────────────────────────
print("\n" + "=" * 78)
print("DECISIÓN")
print("-" * 78)
if not por_anio:
    print(f"  Todas las bases caben cargadas de una vez.")
    print(f"  El paso siguiente trabaja con las tablas completas.")
    TRABAJAR_POR_ANIO = []
else:
    print(f"  Hay que repartir por año: {', '.join(por_anio)}")
    print(f"  Esas bases se procesan año por año y se guarda el resultado de")
    print(f"  cada uno, sin tener nunca todos los años al tiempo. No se pierde")
    print(f"  ningún dato: cambia el orden del trabajo, no su alcance.")
    TRABAJAR_POR_ANIO = por_anio

print(f"\n  Memoria libre ahora : {psutil.virtual_memory().available/1024**3:.1f} GB")
if psutil.virtual_memory().available / 1024**3 < 3:
    print(f"  Conviene cerrar otros programas antes de seguir.")

`CELDA 15`

## Paso 4 · Depurar: nulos, duplicados y valores imposibles

### 4.1 Qué se revisa y por qué

Con los datos ya convertidos, toca mirar qué tan buenos son. Son cuatro revisiones
y cada una busca un problema distinto.

**Cuántos datos faltan.** Después de convertir los `No Definido` en vacío de verdad,
el porcentaje de faltantes sube. Lo que importa no es que suba, sino **dónde**: una
columna con 90 % de vacíos no sirve como variable, por más que exista.

**Si hay filas repetidas.** Un proceso que aparece dos veces se contaría dos veces
en todos los promedios. Como la descarga se hizo mes por mes y cada mes se pidió con
un rango de fechas que no se solapa con el siguiente, no debería haber repetidos.
Esta revisión es la que confirma que ese diseño funcionó.

**Si hay valores imposibles.** No se trata de valores raros, sino de valores que no
pueden existir: un precio negativo, una fecha de adjudicación anterior a la de
publicación, un proceso con menos de cero oferentes.

**Si hay valores extremos.** Estos sí existen y son legítimos, pero distorsionan.
Un contrato de un billón de pesos en medio de miles de contratos de diez millones
arrastra cualquier promedio. No se borran: se identifican, para decidir después qué
hacer con ellos.

### 4.2 Cómo se mide sin llenar la memoria

La revisión se hace **archivo por archivo**, acumulando los conteos. En vez de abrir
los 8,6 millones de filas para contar vacíos, se abre un mes, se cuentan sus vacíos,
se suma al acumulado y se suelta.

El resultado es idéntico al de contar sobre la tabla completa, porque sumar vacíos
es una operación que se

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 16  ·  Revisión de calidad: vacíos, duplicados e imposibles
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Los archivos convertidos de las cuatro bases.
# QUÉ HACE   : Los recorre uno por uno acumulando conteos: cuántos vacíos hay
#              por columna, cuántas llaves repetidas y cuántos valores que no
#              pueden existir. Nunca tiene más de un archivo abierto.
# QUÉ SALE   : Un informe por base con el porcentaje de vacíos de cada columna
#              y el conteo de cada problema encontrado.
# PARA QUÉ   : Para saber con qué se está trabajando antes de construir nada.
#              Esta celda mide y documenta; no borra ni corrige.
# ──────────────────────────────────────────────────────────────────────

# Reglas de lo que no puede existir. Cada una dice, para una base, qué columna
# revisar y qué condición marca un valor imposible.
IMPOSIBLES = {
    "procesos": [
        ("precio_base negativo",
         lambda t: (t["precio_base"] < 0).sum()),
        ("adjudicación anterior a la publicación",
         lambda t: (t["fecha_adjudicacion"] < t["fecha_de_publicacion_del"]).sum()),
        ("oferentes negativos",
         lambda t: (t["proveedores_unicos_con"] < 0).sum()),
        ("adjudicado sin NIT de adjudicatario",
         lambda t: ((t["adjudicado"].astype("string").str.lower() == "si")
                    & t["nit_del_proveedor_adjudicado"].isna()).sum()),
        ("valor adjudicado mayor a 10 veces el precio base",
         lambda t: (t["valor_total_adjudicacion"] > t["precio_base"] * 10).sum()),
    ],
    "contratos": [
        ("valor del contrato negativo",
         lambda t: (t["valor_del_contrato"] < 0).sum()),
        ("pagado mayor que el valor del contrato",
         lambda t: (t["valor_pagado"] > t["valor_del_contrato"] * 1.001).sum()),
        ("fin anterior al inicio",
         lambda t: (t["fecha_de_fin_del_contrato"]
                    < t["fecha_de_inicio_del_contrato"]).sum()),
        ("días adicionados negativos",
         lambda t: (t["dias_adicionados"] < 0).sum()),
    ],
    "proveedores": [
        ("inscripción posterior al corte",
         lambda t: (t["fecha_creacion"] > CORTE).sum()),
    ],
    "grupos": [
        ("participación fuera de 0 a 100",
         lambda t: ((t["participacion"] < 0) | (t["participacion"] > 100)).sum()),
        ("socio inscrito después del grupo",
         lambda t: (t["fecha_creaci_n_participante"]
                    > t["fecha_creaci_n_grupo"]).sum()),
    ],
}

# Qué columna identifica cada fila, para buscar repetidos
LLAVE_UNICA = {
    "procesos": "id_del_proceso",
    "contratos": "id_contrato",
    "proveedores": "codigo",
    "grupos": None,   # una fila es un socio dentro de un grupo: la llave es doble
}


def revisar(base):
    """Recorre los archivos de una base acumulando los conteos de calidad."""
    archivos = sorted((DATOS / base / "convertido").glob("*.parquet"))
    if not archivos:
        return None

    vacios = None          # vacíos acumulados por columna
    filas = 0
    llaves = set()         # para detectar repetidos entre archivos
    repetidos = 0
    problemas = {nombre: 0 for nombre, _ in IMPOSIBLES.get(base, [])}

    llave = LLAVE_UNICA[base]

    for ruta in archivos:
        t = pd.read_parquet(ruta)
        filas += len(t)

        # ── vacíos ──
        v = t.isna().sum()
        vacios = v if vacios is None else vacios.add(v, fill_value=0)

        # ── repetidos ──
        # Se guarda el conjunto de llaves vistas. Para procesos son 8,6 millones
        # de textos, que ocupan memoria, pero mucho menos que la tabla entera y
        # es la única forma de detectar un repetido entre dos meses distintos.
        if llave:
            vistas = t[llave].dropna()
            repetidos += len(vistas) - vistas.nunique()        # dentro del archivo
            nuevas = set(vistas)
            repetidos += len(llaves & nuevas)                  # contra los anteriores
            llaves |= nuevas

        # ── imposibles ──
        for nombre, prueba in IMPOSIBLES.get(base, []):
            try:
                problemas[nombre] += int(prueba(t))
            except Exception as e:
                problemas[nombre] = f"no se pudo medir ({type(e).__name__})"

        del t
        gc.collect()

    return {"filas": filas, "vacios": vacios, "repetidos": repetidos,
            "problemas": problemas, "llaves_unicas": len(llaves) if llave else None}


print("REVISIÓN DE CALIDAD")
print("=" * 78)

calidad = {}
for base in ("proveedores", "grupos", "contratos", "procesos"):
    print(f"\n{base.upper()}")
    print("-" * 78)
    r = revisar(base)
    if r is None:
        print("  sin archivos convertidos")
        continue
    calidad[base] = r

    pct = (r["vacios"] / r["filas"] * 100).sort_values(ascending=False)

    print(f"  Filas: {r['filas']:,}".replace(",", ".")
          + (f"   ·   identificadores distintos: {r['llaves_unicas']:,}"
             .replace(",", ".") if r["llaves_unicas"] else ""))
    print(f"  Filas repetidas: {r['repetidos']:,}".replace(",", ".")
          + ("   correcto" if r["repetidos"] == 0 else "   <-- REVISAR"))

    print(f"\n  VACÍOS POR COLUMNA (solo las que tienen)")
    con_vacios = pct[pct > 0]
    if len(con_vacios) == 0:
        print(f"    ninguna columna tiene vacíos")
    else:
        for col, p in con_vacios.items():
            barra = "█" * int(p / 4)
            aviso = "   inservible" if p > 90 else ("   dudosa" if p > 50 else "")
            print(f"    {col:38} {p:>5.1f} %  {barra}{aviso}")
        sin_vacios = len(pct) - len(con_vacios)
        print(f"    ({sin_vacios} columnas sin ningún vacío)")

    if r["problemas"]:
        print(f"\n  VALORES QUE NO PUEDEN EXISTIR")
        for nombre, n in r["problemas"].items():
            if isinstance(n, str):
                print(f"    {nombre:48} {n}")
            else:
                p = n / r["filas"] * 100
                marca = "correcto" if n == 0 else f"{p:.2f} % de las filas"
                print(f"    {nombre:48} {n:>9,}".replace(",", ".") + f"   {marca}")

print("\n" + "=" * 78)
print("RESUMEN")
print("-" * 78)
print(f"  {'base':14}{'filas':>12}{'columnas':>10}{'con vacíos':>12}"
      f"{'repetidas':>11}{'imposibles':>12}")
print("-" * 78)
for base, r in calidad.items():
    con_v = int((r["vacios"] > 0).sum())
    imp = sum(n for n in r["problemas"].values() if isinstance(n, int))
    print(f"  {base:14}{r['filas']:>12,}".replace(",", ".")
          + f"{len(r['vacios']):>10}{con_v:>12}{r['repetidos']:>11,}".replace(",", ".")
          + f"{imp:>12,}".replace(",", "."))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 17  ·  Qué son los procesos repetidos y qué significa "adjudicado"
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Tres meses de Procesos, de años distintos, suficientes para ver
#              los dos problemas sin cargar los nueve millones de filas.
# QUÉ HACE   : Dos averiguaciones. La primera toma los identificadores que se
#              repiten y compara las filas entre sí, columna por columna, para
#              ver en qué se diferencian. La segunda cruza la columna
#              "adjudicado" contra el estado del procedimiento y contra el NIT
#              del adjudicatario, para ver qué mide cada una.
# QUÉ SALE   : En qué columnas difieren las filas repetidas, y una tabla que
#              muestra qué combinaciones de estado existen de verdad.
# PARA QUÉ   : De la primera depende qué es una fila de la tabla final. De la
#              segunda depende qué se predice. Las dos decisiones centrales de
#              la fase dependen de esta celda.
# ──────────────────────────────────────────────────────────────────────

# Tres meses de años distintos, para no sacar conclusiones de una temporada
# particular ni de un único momento de la plataforma.
muestras_ruta = [
    DATOS / "procesos" / "convertido" / "procesos_2018-06.parquet",
    DATOS / "procesos" / "convertido" / "procesos_2021-06.parquet",
    DATOS / "procesos" / "convertido" / "procesos_2025-06.parquet",
]
pm = pd.concat([pd.read_parquet(r) for r in muestras_ruta if r.exists()],
               ignore_index=True)

print(f"Muestra: {len(pm):,} filas de tres meses".replace(",", "."))
print(f"Identificadores distintos: {pm['id_del_proceso'].nunique():,}"
      .replace(",", "."))
rep = len(pm) - pm["id_del_proceso"].nunique()
print(f"Repetidos: {rep:,}".replace(",", ".")
      + f"   ({rep/len(pm)*100:.1f} % de la muestra)\n")


# ══ AVERIGUACIÓN 1 · En qué se diferencian las filas repetidas ══════════
print("=" * 78)
print("1 · QUÉ DISTINGUE A LAS FILAS QUE COMPARTEN IDENTIFICADOR")
print("=" * 78)

veces = pm["id_del_proceso"].value_counts()
repetidos = veces[veces > 1]

print(f"  Identificadores que aparecen más de una vez: {len(repetidos):,}"
      .replace(",", "."))
print(f"  Cuántas filas genera cada uno:")
reparto = repetidos.value_counts().sort_index()
print(f"    de 2 a 5 filas   : {int(reparto.loc[2:5].sum()):,} procesos"
      .replace(",", "."))
print(f"    de 6 a 20 filas  : {int(reparto.loc[6:20].sum()):,} procesos"
      .replace(",", "."))
print(f"    más de 20 filas  : {int(reparto.loc[21:].sum()):,} procesos"
      .replace(",", "."))
print(f"    el mayor         : {int(repetidos.max()):,} filas de un solo proceso"
      .replace(",", "."))

if len(repetidos):
    duplicadas = pm[pm["id_del_proceso"].isin(repetidos.index)]

    # Para cada columna: en cuántos de esos grupos el valor cambia entre filas
    print(f"\n  COLUMNAS QUE CAMBIAN ENTRE LAS FILAS DE UN MISMO PROCESO")
    print(f"  (de {len(repetidos):,} grupos repetidos)".replace(",", "."))
    print("  " + "-" * 74)

    cambian = {}
    g = duplicadas.groupby("id_del_proceso", observed=True)
    for col in duplicadas.columns:
        if col == "id_del_proceso":
            continue
        try:
            cambian[col] = int((g[col].nunique(dropna=False) > 1).sum())
        except Exception:
            cambian[col] = -1

    serie = pd.Series(cambian).sort_values(ascending=False)
    for col, n in serie[serie > 0].items():
        p = n / len(repetidos) * 100
        print(f"    {col:34} {n:>7,}".replace(",", ".")
              + f"  {p:>5.1f} %  " + "█" * int(p / 4))

    iguales = serie[serie == 0]
    print(f"\n    ({len(iguales)} columnas tienen el mismo valor en todas las"
          f" filas de cada grupo)")

    # ── Un ejemplo, recortado ────────────────────────────────────────
    # Se muestra el caso más extremo, que es donde el patrón se ve claro, pero
    # solo las primeras diez filas: imprimirlo entero infla el archivo del
    # cuaderno sin agregar nada que no se vea en diez.
    ejemplo_id = repetidos.index[0]
    ej = duplicadas[duplicadas["id_del_proceso"] == ejemplo_id]
    varian = [c for c in ej.columns if ej[c].nunique(dropna=False) > 1]

    print(f"\n  EJEMPLO · proceso {ejemplo_id}")
    print(f"  Genera {len(ej):,} filas".replace(",", ".")
          + f" y solo cambian {len(varian)} columnas: {varian}")

    if varian:
        print(ej[["id_del_proceso"] + varian[:5]].head(10).to_string(index=False))
        if len(ej) > 10:
            print(f"    … y {len(ej)-10:,} filas más del mismo proceso"
                  .replace(",", "."))

        # La prueba de que es un producto cruzado: si las filas fueran
        # adjudicaciones distintas, el número de filas sería igual al número de
        # proveedores. Si es un cruce, es el producto de los valores distintos
        # de cada columna.
        combinaciones = 1
        for c in varian[:2]:
            combinaciones *= ej[c].nunique(dropna=False)
        print(f"\n    Valores distintos en las dos primeras columnas que cambian:"
              f" {' x '.join(str(ej[c].nunique(dropna=False)) for c in varian[:2])}"
              f" = {combinaciones:,}".replace(",", "."))
        print(f"    Filas reales del proceso: {len(ej):,}".replace(",", "."))
        print(f"    Si los dos números se parecen, la fuente está guardando")
        print(f"    todas las combinaciones posibles y no adjudicaciones reales.")


# ══ AVERIGUACIÓN 2 · Qué significa "adjudicado" ═════════════════════════
print("\n" + "=" * 78)
print("2 · QUÉ MIDE LA COLUMNA 'ADJUDICADO'")
print("=" * 78)

print(f"\n  REPARTO DE 'adjudicado'")
for v, n in pm["adjudicado"].value_counts(dropna=False).items():
    print(f"    {str(v):20} {n:>9,}".replace(",", ".")
          + f"   {n/len(pm)*100:>5.1f} %")

print(f"\n  REPARTO DE 'estado_del_procedimiento'")
for v, n in pm["estado_del_procedimiento"].value_counts(dropna=False).head(12).items():
    print(f"    {str(v)[:38]:40} {n:>9,}".replace(",", ".")
          + f"   {n/len(pm)*100:>5.1f} %")

print(f"\n  REPARTO DE 'estado_resumen'")
for v, n in pm["estado_resumen"].value_counts(dropna=False).head(12).items():
    print(f"    {str(v)[:38]:40} {n:>9,}".replace(",", ".")
          + f"   {n/len(pm)*100:>5.1f} %")

print(f"\n  CRUCE · adjudicado contra estado del procedimiento")
print(pd.crosstab(pm["estado_del_procedimiento"], pm["adjudicado"],
                  dropna=False).to_string())

print(f"\n  ¿TIENEN NIT DE ADJUDICATARIO?")
print(pd.crosstab(pm["adjudicado"], pm["nit_del_proveedor_adjudicado"].notna(),
                  rownames=["adjudicado"],
                  colnames=["tiene NIT de adjudicatario"]).to_string())

print(f"\n  POR MODALIDAD · cuántos procesos y qué porcentaje tiene adjudicatario")
print(f"  Esta es la tabla que decide el alcance del análisis.")
print("  " + "-" * 74)
porc = (pm.groupby("modalidad_de_contratacion", observed=True)
          .agg(procesos=("id_del_proceso", "size"),
               con_nit=("nit_del_proveedor_adjudicado",
                        lambda s: s.notna().mean() * 100))
          .query("procesos >= 50").sort_values("con_nit", ascending=False))
for nombre, fila in porc.iterrows():
    marca = "" if fila["con_nit"] > 1 else "   <-- sin competencia"
    print(f"    {str(nombre)[:44]:46} {int(fila['procesos']):>8,}".replace(",", ".")
          + f"   {fila['con_nit']:>5.1f} %{marca}")

del pm, duplicadas
gc.collect()

`CELDA 18`

## Paso 5 · Qué entra en la tabla final

Las revisiones de los pasos anteriores dejaron tres problemas, y cada uno se
resuelve con un filtro. Conviene justificarlos antes de aplicarlos, porque entre los
tres descartan la mayor parte de las filas y eso tiene que estar sustentado.

### 5.1 Filtro uno: solo modalidades donde se compite

La columna `modalidad_de_contratacion` separa dos mundos distintos.

| Modalidad | Qué es |
|---|---|
| Contratación directa | La entidad escoge a quién contratar sin convocatoria |
| Contratación régimen especial | Entidades con régimen propio, que contratan sin licitar |
| Solicitud de información a los Proveedores | Un estudio de mercado, no un proceso |
| Subasta de prueba | Ensayos del sistema |

En esas cuatro, el porcentaje de procesos con adjudicatario registrado es **cero**.
No es un dato faltante: es que no hay adjudicación en el sentido de una competencia.
Nadie se presenta porque no hay a qué presentarse.

Y son la mayoría: en los años recientes llegan al 85 % de todos los procesos.

Para un análisis dirigido a una empresa que quiere licitar, incluirlas sería un
error de planteamiento. El modelo aprendería a responder "no se adjudica" el 90 % de
las veces, acertaría, y no diría nada útil. Peor: diluiría la señal de los procesos
donde sí hay competencia, que son los únicos que le interesan al usuario.

**Lo que se conserva** son todas las modalidades donde hay convocatoria abierta:
licitación pública, mínima cuantía, selección abreviada en sus variantes, concurso
de méritos, acuerdos marco, enajenación de bienes, y las dos variantes "con ofertas"
de contratación directa y régimen especial, que a pesar del nombre sí reciben
propuestas.

### 5.2 Filtro dos: solo procesos que ya terminaron

La variable objetivo es si el proceso se adjudicó. Eso solo se sabe cuando el
proceso cerró.

| Estado | Qué significa | ¿Entra? |
|---|---|---|
| Seleccionado | El proceso concluyó | Sí |
| Cancelado | El proceso concluyó sin adjudicar | Sí |
| Publicado | Está abierto, recibiendo propuestas | No |
| Evaluación | Está decidiendo | No |
| Abierto, Borrador, Suspendido, Aprobado | Estados intermedios o residuales | No |

Un proceso en `Publicado` no es un proceso que no se adjudicó: es un proceso cuyo
desenlace todavía no ocurrió. Contarlo como "no adjudicado" sería inventar un
resultado.

Este filtro pesa mucho más en los meses recientes, donde hay muchos procesos en
curso, y casi nada en 2017 o 2018, donde ya todo concluyó. Esa diferencia hay que
tenerla presente: el filtro recorta el final de la ventana más que el principio.

### 5.3 Filtro tres: una fila por proceso

La fuente duplica los procesos con varias adjudicaciones, y lo hace de una forma
particular: guarda **todas las combinaciones posibles** entre los proveedores
adjudicados y los valores adjudicados. Un proceso con 40 adjudicatarios y 40 valores
aparece 1.600 veces.

Eso significa que en esos procesos **no se puede saber qué proveedor se ganó cuál
valor**. Esa información se perdió en el origen, no en esta descarga.

Las 27 columnas que describen el proceso —entidad, modalidad, precio base, fechas,
competencia— son idénticas en todas las filas del grupo. Así que quedarse con una
fila por proceso no pierde nada de lo que el análisis necesita.

Lo que sí se conserva del grupo es **cuántos adjudicatarios distintos tuvo**, que se
cuenta antes de descartar las filas sobrantes. Es la única información real que
aportaba la duplicación.

### 5.4 Lo que queda, y lo que hay que declarar

Después de los tres filtros, el análisis deja de hablar de toda la contratación
pública y pasa a hablar de **la contratación que se gana compitiendo**. Para el
usuario que el proyecto definió —la empresa que decide a qué presentarse— esa es
exactamente la pregunta.

Es un recorte grande y hay que decirlo sin rodeos en el informe: de cada diez
procesos publicados en SECOP II, solo uno o dos admiten competencia. El resto se
adjudica sin convocatoria. Ese hallazgo, por sí solo, ya le dice algo a quien vive
de licitar.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 19  ·  Aplicar los tres filtros y construir la base de la tabla
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Los 117 archivos convertidos de Procesos, nueve millones de
#              filas en total.
# QUÉ HACE   : Los recorre uno por uno aplicando los tres filtros del paso 5
#              —modalidades competitivas, procesos terminados, una fila por
#              proceso— y va contando cuántas filas sobreviven a cada uno.
#              Trabaja archivo por archivo porque los nueve millones sin
#              filtrar no caben en la memoria de este equipo.
# QUÉ SALE   : La tabla base del análisis, con una fila por proceso
#              competitivo y terminado, y un informe de cuánto recortó cada
#              filtro, año por año.
# PARA QUÉ   : Es el momento en que el conjunto deja de ser "todo lo que hay"
#              y pasa a ser "lo que responde la pregunta". Todo lo que sigue
#              se construye sobre esta tabla.
# ──────────────────────────────────────────────────────────────────────

# ── Filtro 1 · modalidades ────────────────────────────────────────────
# Se nombran las que NO entran, que son cuatro, en vez de las que sí, que son
# una docena y pueden aparecer escritas de formas distintas según el año. Así
# una modalidad nueva o con otra grafía entra por defecto, que es el
# comportamiento seguro: es preferible revisar algo de más que perder procesos
# sin darse cuenta.
SIN_COMPETENCIA = {
    "Contratación directa",
    "Contratación régimen especial",
    "Solicitud de información a los Proveedores",
    "Subasta de prueba",
}

# ── Filtro 2 · estados ────────────────────────────────────────────────
TERMINADOS = {"Seleccionado", "Cancelado"}

# ── Columnas que se conservan ─────────────────────────────────────────
# Quedan afuera subtipo_de_contrato, que está vacía en el 100 % de los casos,
# y el departamento y la ciudad del proveedor adjudicado, que además de estar
# casi siempre vacías son de las que el producto cruzado ensucia.
#
# El NIT del adjudicatario sí se conserva. Medido sobre los nueve millones
# aparecía vacío en el 93 %, pero esa cifra estaba dominada por la contratación
# directa, que nunca lo registra. Entre los procesos competitivos y terminados
# está presente en la mayoría, y es la llave con que se pega el perfil del
# proveedor que ganó.
COLUMNAS_BASE = [
    # llaves
    "id_del_proceso", "nit_entidad", "codigo_entidad",
    # quién convoca
    "entidad", "departamento_entidad", "ciudad_entidad", "ordenentidad",
    # qué se contrata
    "modalidad_de_contratacion", "tipo_de_contrato",
    "codigo_principal_de_categoria", "fase", "duracion", "unidad_de_duracion",
    "precio_base", "numero_de_lotes",
    # cuándo
    "fecha_de_publicacion_del", "fecha_de_recepcion_de",
    # competencia
    "proveedores_invitados", "proveedores_con_invitacion", "visualizaciones_del",
    "proveedores_que_manifestaron", "respuestas_al_procedimiento",
    "respuestas_externas", "conteo_de_respuestas_a_ofertas",
    "proveedores_unicos_con",
    # resultado · ninguna de estas puede ser variable explicativa
    "adjudicado", "estado_del_procedimiento", "estado_resumen",
    "valor_total_adjudicacion", "fecha_adjudicacion",
    "nit_del_proveedor_adjudicado",
]


def filtrar_archivo(ruta):
    """Aplica los tres filtros a un mes y devuelve las filas que sobreviven.

    Entra: la ruta de un archivo mensual convertido.
    Sale:  la tabla filtrada, y el conteo de cuánto quitó cada filtro.

    El orden importa por memoria: primero se descartan filas, después se
    recortan columnas. Al revés habría que conservar columnas de filas que de
    todos modos se van a botar.
    """
    t = pd.read_parquet(ruta)
    conteo = {"crudas": len(t)}

    # filtro 1 · modalidades donde se compite
    t = t[~t["modalidad_de_contratacion"].astype("string").isin(SIN_COMPETENCIA)]
    conteo["competitivas"] = len(t)

    # filtro 2 · procesos que ya terminaron
    t = t[t["estado_del_procedimiento"].astype("string").isin(TERMINADOS)]
    conteo["terminadas"] = len(t)

    if len(t) == 0:
        conteo["unicas"] = 0
        return None, conteo

    # Antes de descartar las filas repetidas se rescata lo único que esa
    # duplicación aportaba: cuántos adjudicatarios distintos tuvo el proceso.
    # En los procesos con varios, la fila que queda conserva el NIT de uno de
    # ellos; cuál no se puede saber, porque la fuente no lo dice.
    adjudicatarios = (t.groupby("id_del_proceso", observed=True)
                       ["nit_del_proveedor_adjudicado"]
                       .nunique(dropna=True)
                       .rename("n_adjudicatarios"))

    # filtro 3 · una fila por proceso
    # Se ordena antes de deduplicar para que la fila que quede sea una con NIT
    # de adjudicatario, cuando el proceso tenga alguna. Sin ordenar podría
    # quedarse con una fila vacía de un proceso que sí se adjudicó.
    t = t.sort_values("nit_del_proveedor_adjudicado", na_position="last")
    t = t.drop_duplicates(subset="id_del_proceso", keep="first")
    conteo["unicas"] = len(t)

    t = t[[c for c in COLUMNAS_BASE if c in t.columns]]
    t = t.merge(adjudicatarios, left_on="id_del_proceso", right_index=True,
                how="left")
    t["anio"] = t["fecha_de_publicacion_del"].dt.year

    return t, conteo


archivos = sorted((DATOS / "procesos" / "convertido").glob("*.parquet"))

print(f"APLICANDO LOS TRES FILTROS A {len(archivos)} ARCHIVOS")
print("=" * 78)

partes, conteos = [], []
reloj = time.perf_counter()

for i, ruta in enumerate(archivos, 1):
    t, c = filtrar_archivo(ruta)
    c["archivo"] = ruta.stem
    conteos.append(c)
    if t is not None and len(t):
        partes.append(t)
    if i % 20 == 0 or i == len(archivos):
        vivas = sum(len(p) for p in partes)
        print(f"   {i:>3}/{len(archivos)}   {vivas:>9,} procesos en pie"
              .replace(",", ".") + f"   {time.perf_counter()-reloj:>5.0f} s")

procesos = pd.concat(partes, ignore_index=True)
del partes
gc.collect()

# Las etiquetas se vuelven a fijar después de unir, porque al concatenar tablas
# con categorías distintas pandas las convierte a texto suelto.
for c in ("entidad", "departamento_entidad", "ciudad_entidad", "ordenentidad",
          "modalidad_de_contratacion", "tipo_de_contrato", "fase",
          "unidad_de_duracion", "adjudicado", "estado_del_procedimiento",
          "estado_resumen", "codigo_principal_de_categoria"):
    if c in procesos:
        procesos[c] = procesos[c].astype("category")

# La variable objetivo, construida acá para poder medirla de una vez. Se
# formaliza en el paso 7.
procesos["adjudicado_si"] = (procesos["adjudicado"].astype("string")
                             .str.lower().eq("si").fillna(False))


# ── Cuánto quitó cada filtro ──────────────────────────────────────────
cnt = pd.DataFrame(conteos)
total = cnt[["crudas", "competitivas", "terminadas", "unicas"]].sum()

print("\n" + "=" * 78)
print("EFECTO DE CADA FILTRO")
print("-" * 78)
print(f"  {'etapa':34}{'procesos':>12}{'queda':>9}{'quita':>9}")
print("-" * 78)
etapas = [
    ("Descargados", total["crudas"], None),
    ("Modalidades donde se compite", total["competitivas"], total["crudas"]),
    ("Que ya terminaron", total["terminadas"], total["competitivas"]),
    ("Una fila por proceso", total["unicas"], total["terminadas"]),
]
for nombre, n, antes in etapas:
    pct = n / total["crudas"] * 100
    quita = f"{(1 - n/antes)*100:>7.1f} %" if antes else "       —"
    print(f"  {nombre:34}{int(n):>12,}".replace(",", ".")
          + f"{pct:>8.1f} %{quita}")

print("-" * 78)
print(f"  La tabla base queda con {len(procesos):,} procesos".replace(",", ".")
      + f" y {procesos.shape[1]} columnas")
print(f"  Ocupa {procesos.memory_usage(deep=True).sum()/1024**2:.0f} MB en memoria")

con_nit = procesos["nit_del_proveedor_adjudicado"].notna()
print(f"\n  Con NIT de adjudicatario : {int(con_nit.sum()):,}".replace(",", ".")
      + f"   ({con_nit.mean()*100:.1f} %)")
print(f"  Adjudicados según la columna 'adjudicado' : "
      f"{int(procesos['adjudicado_si'].sum()):,}".replace(",", ".")
      + f"   ({procesos['adjudicado_si'].mean()*100:.1f} %)")
print(f"  Las dos cifras deben parecerse: miden lo mismo por dos caminos.")


# ── Cómo quedó repartido por año ──────────────────────────────────────
print(f"\nPROCESOS COMPETITIVOS Y TERMINADOS, POR AÑO")
print("-" * 78)
print(f"  {'año':>6}{'procesos':>12}{'adjudicados':>14}{'tasa':>9}   reparto")
print("-" * 78)

por_anio = (procesos.groupby("anio")
                    .agg(n=("id_del_proceso", "size"),
                         adj=("adjudicado_si", "sum")))
por_anio["tasa"] = por_anio["adj"] / por_anio["n"] * 100

mayor = por_anio["n"].max()
for anio, fila in por_anio.iterrows():
    barra = "█" * int(fila["n"] / mayor * 30)
    nota = "  (año incompleto)" if anio == 2026 else ""
    print(f"  {int(anio):>6}{int(fila['n']):>12,}".replace(",", ".")
          + f"{int(fila['adj']):>14,}".replace(",", ".")
          + f"{fila['tasa']:>8.1f} %   {barra}{nota}")

print("-" * 78)
print(f"  {'TOTAL':>6}{len(procesos):>12,}".replace(",", ".")
      + f"{int(procesos['adjudicado_si'].sum()):>14,}".replace(",", ".")
      + f"{procesos['adjudicado_si'].mean()*100:>8.1f} %")

print(f"\n  Memoria libre: {psutil.virtual_memory().available/1024**3:.1f} GB")

`CELDA 20`

## Paso 6 · Integrar las cuatro bases

La tabla base tiene 626.384 procesos y describe bien **qué** se contrata y **quién**
lo convoca. Le falta el contexto que convierte esa descripción en una herramienta de
decisión: cómo se comporta esa entidad, y quién es el proveedor que terminó ganando.

Ese contexto está en las otras tres bases, y cada una entra de una forma distinta.

### 6.1 Por qué no se unen de frente

La tentación es cruzar las cuatro tablas por sus llaves y quedarse con el resultado.
No funciona, por dos razones.

**Las unidades no coinciden.** Procesos tiene una fila por proceso; Contratos tiene
una por contrato, y un proceso puede generar varios; Grupos tiene una por socio
dentro de un consorcio. Cruzar sin más multiplicaría las filas y volvería a crear el
mismo problema que el filtro tres acaba de resolver.

**No cabría en memoria.** Contratos tiene 5,6 millones de filas. Unirla entera a la
tabla base, en un equipo con menos de un giga libre, no termina.

### 6.2 Cómo se hace entonces

Cada base auxiliar se **reduce primero** a una fila por la llave con que se va a
cruzar, y solo después se une. Eso convierte millones de filas en unos pocos miles.

| Base | Se reduce a | Pasa de | A unas |
|---|---|---|---|
| Contratos | Una fila por entidad | 5.642.665 | 15.000 |
| Proveedores | Una fila por NIT | 1.618.521 | 1.618.521, pero se cruzan pocas |
| Grupos | Una fila por NIT de socio | 711.140 | 300.000 |

La reducción no es solo un truco de memoria: es lo que convierte un registro de
hechos en una característica. "Esta entidad firmó 400 contratos y pagó en promedio
el 85 % de lo pactado" es una variable; las 400 filas de esos contratos no lo son.

### 6.3 Qué se le saca a cada base

**De Contratos, el comportamiento de la entidad como cliente.** Es la pieza que le
dice a la empresa con quién conviene meterse, y la que lleva el componente de
control del proyecto.

| Variable | Qué mide | Por qué le importa a quien licita |
|---|---|---|
| Contratos firmados | Volumen de la entidad | Una entidad que contrata poco ofrece pocas oportunidades |
| Proporción pagada | Cuánto del contrato termina pagándose | Riesgo de flujo de caja |
| Días adicionados promedio | Cuánto se prorrogan sus contratos | El plazo real no es el del pliego |
| Desviación de lo facturado | Qué tanto se aleja lo facturado de lo pactado | Señal de planeación floja o de alcance mal definido |
| Valor mediano del contrato | El tamaño típico de lo que firma | Si encaja con la capacidad de la empresa |

**De Proveedores, quién es el que gana.** Antigüedad, si es pyme, dónde está, qué
tipo de empresa es. Permite preguntarse si en esa entidad ganan empresas parecidas a
la mía o muy distintas.

**De Grupos, si gana solo o asociado.** En cuántos consorcios participa un
proveedor, y con qué participación promedio. En procesos grandes, presentarse solo
puede ser la razón de perder.

### 6.4 Las tres uniones y por qué cada una es como es

| Unión | Tipo | Qué pasa con lo que no cruza |
|---|---|---|
| Procesos con perfil de entidad | Por la izquierda | El proceso se conserva, con el perfil vacío. Una entidad sin contratos firmados es información, no un error |
| Procesos con perfil de proveedor | Por la izquierda | Solo cruzan los procesos adjudicados, que son el 82,5 %. Los desiertos no tienen adjudicatario y es correcto que queden vacíos |
| Procesos con experiencia en consorcios | Por la izquierda | La mayoría de proveedores no aparece en ningún grupo. Ese vacío significa "nunca se ha asociado", no "falta el dato" |

Las tres son por la izquierda a propósito: **ninguna puede eliminar un proceso**. La
tabla final tiene que seguir teniendo 626.384 filas al terminar la integración, y
esa es la primera comprobación que se hace después de unir.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 21  ·  Descargar la llave que une Procesos con Contratos
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Nada local. Se le pide al portal una tabla de dos columnas.
# QUÉ HACE   : Descarga la correspondencia entre los dos identificadores que
#              SECOP II le da al mismo proceso. La tabla de Procesos lo llama
#              "id_del_proceso" y usa el prefijo CO1.REQ; la de Contratos lo
#              llama "proceso_de_compra" y usa CO1.BDOS. Son el mismo proceso
#              con dos códigos, y la única columna que los relaciona es
#              "id_del_portafolio", que no se pidió en la descarga original.
# QUÉ SALE   : Un archivo por mes con las dos columnas, y la tabla de
#              correspondencia cargada en memoria.
# PARA QUÉ   : Sin esta llave las dos tablas principales no se pueden cruzar.
#              Se baja solo para las modalidades competitivas, que son las que
#              quedaron en la tabla base: un millón de filas en vez de nueve.
# ──────────────────────────────────────────────────────────────────────

# El mismo filtro de modalidades del paso 5, escrito en el lenguaje de
# consulta del portal.
FILTRO_COMPETITIVAS = ("modalidad_de_contratacion not in ("
                       + ",".join(f"'{m}'" for m in sorted(SIN_COMPETENCIA))
                       + ")")

print("Filtro que se le manda al portal:")
print(f"  {FILTRO_COMPETITIVAS}\n")


def bajar_mapa_mes(anio, mes, limite=50000):
    """Baja la correspondencia de identificadores de un mes."""
    carpeta = DATOS / "mapa"
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / f"mapa_{anio}-{mes:02d}.csv"

    if destino.exists():
        return "ya estaba", contar_filas(destino, "id_del_proceso")

    desde = pd.Timestamp(anio, mes, 1)
    hasta = min(desde + pd.offsets.MonthBegin(1), CORTE + pd.Timedelta(days=1))
    if desde > CORTE:
        return "fuera del corte", 0

    campo = BASES["procesos"]["fecha"]
    donde = (f"{campo} >= '{desde:%Y-%m-%d}' AND {campo} < '{hasta:%Y-%m-%d}'"
             f" AND {FILTRO_COMPETITIVAS}")

    partes = bajar_todas_las_tandas(
        BASES["procesos"]["codigo"],
        ["id_del_proceso", "id_del_portafolio"],
        donde, limite)

    temporal = carpeta / f"mapa_{anio}-{mes:02d}.parcial"
    if partes:
        t = pd.concat(partes, ignore_index=True)
        t.to_csv(temporal, index=False, encoding="utf-8")
        filas = len(t)
    else:
        temporal.write_text("id_del_proceso,id_del_portafolio\n", encoding="utf-8")
        filas = 0
    temporal.replace(destino)
    return "descargado", filas


meses = meses_del_periodo(INICIO, CORTE)
print(f"DESCARGANDO LA LLAVE · {len(meses)} meses")
print("=" * 78)

reloj, total, bajados = time.perf_counter(), 0, 0
for i, (anio, mes) in enumerate(meses, 1):
    estado, filas = bajar_mapa_mes(anio, mes)
    total += filas
    if estado == "descargado":
        bajados += 1
    if i % 15 == 0 or i == len(meses):
        falta = (len(meses) - i) * ((time.perf_counter() - reloj) / max(bajados, 1))
        print(f"   {i:>3}/{len(meses)}   {total:>9,} parejas".replace(",", ".")
              + f"   {time.perf_counter()-reloj:>5.0f} s"
              + (f"   faltan ~{falta/60:.0f} min" if i < len(meses) else ""))

# ── Se arma la tabla de correspondencia ───────────────────────────────
mapa = pd.concat(
    [pd.read_csv(r, dtype=str, keep_default_na=False, na_values=[""])
     for r in sorted((DATOS / "mapa").glob("mapa_*.csv"))],
    ignore_index=True)

mapa = mapa.dropna(subset=["id_del_proceso", "id_del_portafolio"])
mapa = mapa.drop_duplicates(subset="id_del_proceso")

print("\n" + "=" * 78)
print("TABLA DE CORRESPONDENCIA")
print("-" * 78)
print(f"  Parejas de identificadores : {len(mapa):,}".replace(",", "."))
print(f"  Procesos en la tabla base  : {len(procesos):,}".replace(",", "."))

# ── Se pega a la tabla base y se mide la cobertura ────────────────────
procesos = procesos.merge(mapa, on="id_del_proceso", how="left")
cobertura = procesos["id_del_portafolio"].notna()

print(f"\n  Procesos de la base con llave : {int(cobertura.sum()):,}"
      .replace(",", ".") + f"   ({cobertura.mean()*100:.1f} %)")

if cobertura.mean() < 0.95:
    print(f"\n  Aviso: la cobertura debería estar cerca del 100 %. Si no lo está,")
    print(f"  revisar que el filtro de modalidades sea idéntico al del paso 5.")

print(f"\n  Ejemplo de la correspondencia:")
print(procesos.loc[cobertura, ["id_del_proceso", "id_del_portafolio",
                               "entidad"]].head(5).to_string(index=False))

print(f"\n  Memoria libre: {psutil.virtual_memory().available/1024**3:.1f} GB")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 22  ·  Perfil de la entidad y recuperación del adjudicatario
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : Los archivos de Contratos y la tabla base de procesos, que ahora
#              sí tiene la llave para cruzarlas.
# QUÉ HACE   : Tres cosas. En un solo recorrido de los archivos de Contratos
#              resume el comportamiento de cada entidad como cliente y recupera
#              quién ganó cada proceso. Aparte, calcula año por año cómo se
#              había comportado cada entidad hasta el año anterior.
# QUÉ SALE   : Ocho variables de contexto pegadas a cada proceso, el
#              adjudicatario, y la comprobación de que el perfil sirve.
# PARA QUÉ   : El perfil de entidad es lo que le dice a la empresa con quién
#              conviene meterse. El adjudicatario sirve para describir qué tipo
#              de empresa gana dónde, no para predecir, porque solo se conoce
#              cuando el proceso ya cerró.
#
#              La ventana hacia atrás es la diferencia entre un conjunto que
#              sirve y uno que produce un modelo falsamente brillante. Calcular
#              la tasa de adjudicación de una entidad con todos sus procesos,
#              incluido el que se quiere predecir, es contarle la respuesta al
#              modelo.
# ──────────────────────────────────────────────────────────────────────

# ══ 1 · UN SOLO RECORRIDO DE CONTRATOS ══════════════════════════════════
# La llave del cruce es el identificador de portafolio, no el del proceso.
portafolios = set(procesos["id_del_portafolio"].dropna())
print(f"Portafolios a los que hay que buscarles contrato: {len(portafolios):,}"
      .replace(",", "."))

COLS_CONTRATOS = ["nit_entidad", "proceso_de_compra", "documento_proveedor",
                  "valor_del_contrato", "valor_pagado", "dias_adicionados",
                  "fecha_de_firma"]

archivos_c = sorted((DATOS / "contratos" / "convertido").glob("*.parquet"))

print(f"\nRECORRIENDO {len(archivos_c)} ARCHIVOS DE CONTRATOS")
print("=" * 78)

# Los dos acumuladores. El perfil se construye con sumas y conteos porque son
# las únicas operaciones que se pueden ir acumulando archivo por archivo: la
# suma de dos meses es la suma de las sumas. Una mediana no se puede partir
# así, y por eso las proporciones se calculan al final dividiendo totales.
#
# Sobre "valor_pagado": no mide lo que la entidad pagó, mide si alguien
# registró el pago en la plataforma. Los contratos viejos aparecen con menos
# pagos que los nuevos, que es lo contrario de lo que haría una curva de pagos
# real. Por eso la variable se llama "reporta pagos" y no "pagó": lo que
# describe es el hábito administrativo de la entidad, no su cumplimiento.
acum_c, enlaces = [], []
reloj = time.perf_counter()

for i, ruta in enumerate(archivos_c, 1):
    t = pd.read_parquet(ruta, columns=COLS_CONTRATOS)

    # ── parte 1 · sumas por entidad ──
    t["_uno"] = 1
    t["_con_pago"] = (t["valor_pagado"] > 0).astype("int8")
    acum_c.append(t.groupby("nit_entidad", observed=True).agg(
        contratos=("_uno", "sum"),
        valor_sum=("valor_del_contrato", "sum"),
        con_pago=("_con_pago", "sum"),
        dias_sum=("dias_adicionados", "sum"),
        dias_n=("dias_adicionados", "count")))

    # ── parte 2 · qué contrato salió de cada proceso ──
    # Solo se guardan los contratos cuyo portafolio está en la tabla base. Sin
    # ese filtro habría que cargar en memoria los 5,6 millones de parejas.
    e = t.loc[t["proceso_de_compra"].isin(portafolios),
              ["proceso_de_compra", "documento_proveedor",
               "valor_del_contrato", "fecha_de_firma"]]
    if len(e):
        enlaces.append(e)

    del t
    if i % 30 == 0 or i == len(archivos_c):
        print(f"   {i:>3}/{len(archivos_c)}   "
              f"{sum(len(x) for x in enlaces):>8,} contratos enlazados"
              .replace(",", ".") + f"   {time.perf_counter()-reloj:>5.0f} s")

gc.collect()


# ══ 2 · LO QUE APORTA CONTRATOS AL PERFIL ═══════════════════════════════
# Estas cuatro no se pueden partir por año sin volver a recorrer los archivos,
# así que se dejan como perfil general de la entidad. Es una concesión y queda
# declarada: para un proceso de 2018, el conteo de contratos de su entidad
# incluye contratos posteriores.
#
# El riesgo es bajo porque ninguna de las cuatro se relaciona de forma directa
# con que un proceso particular se adjudique: son descriptores de tamaño y de
# hábito administrativo, no del desenlace.
ent_c = pd.concat(acum_c).groupby(level=0).sum()
del acum_c
gc.collect()

ent_c["ent_contratos"]         = ent_c["contratos"]
ent_c["ent_valor_medio"]       = ent_c["valor_sum"] / ent_c["contratos"]
ent_c["ent_reporta_pagos_pct"] = ent_c["con_pago"] / ent_c["contratos"]
ent_c["ent_dias_adicionados"]  = ent_c["dias_sum"] / ent_c["dias_n"].replace(0, np.nan)
ent_c = ent_c[["ent_contratos", "ent_valor_medio", "ent_reporta_pagos_pct",
               "ent_dias_adicionados"]]

print(f"\n  Entidades con contratos firmados: {len(ent_c):,}".replace(",", "."))


# ══ 3 · QUIÉN GANÓ CADA PROCESO ═════════════════════════════════════════
enl = pd.concat(enlaces, ignore_index=True)
del enlaces
gc.collect()

print(f"\nCONTRATOS QUE SALIERON DE LOS PROCESOS DE LA BASE")
print("-" * 78)
print(f"  Contratos enlazados: {len(enl):,}".replace(",", "."))

# Un proceso con varios lotes genera varios contratos. Se conserva el del
# contrato de mayor valor, que es el adjudicatario principal, y aparte se
# cuenta cuántos proveedores distintos quedaron contratados.
enl = enl.sort_values("valor_del_contrato", ascending=False)
principal = (enl.drop_duplicates(subset="proceso_de_compra", keep="first")
                .set_index("proceso_de_compra")
                [["documento_proveedor", "valor_del_contrato", "fecha_de_firma"]]
                .rename(columns={"documento_proveedor": "nit_ganador_contrato",
                                 "valor_del_contrato": "valor_contrato_mayor",
                                 "fecha_de_firma": "fecha_primer_contrato"}))
cuantos = (enl.groupby("proceso_de_compra")["documento_proveedor"]
              .nunique().rename("n_contratistas"))

print(f"  Procesos con al menos un contrato: {len(principal):,}"
      .replace(",", ".") + f"   ({len(principal)/len(procesos)*100:.1f} % de la base)")

del enl
gc.collect()


# ══ 4 · EL PERFIL AÑO POR AÑO, SIN MIRAR EL FUTURO ══════════════════════
print(f"\nPERFIL DE ENTIDAD CON VENTANA HACIA ATRÁS")
print("=" * 78)

# Primero se resume la actividad año por año. Después esos resúmenes se van
# acumulando hacia adelante, de modo que el acumulado de 2020 contiene 2017,
# 2018, 2019 y 2020 — y es el que le corresponde a un proceso de 2021.
por_ent_anio = (procesos.groupby(["nit_entidad", "anio"], observed=True)
                        .agg(n=("id_del_proceso", "size"),
                             adj=("adjudicado_si", "sum"),
                             oferentes=("proveedores_unicos_con", "sum"),
                             oferentes_n=("proveedores_unicos_con", "count"),
                             precio=("precio_base", "sum"),
                             precio_n=("precio_base", "count"))
                        .reset_index())

print(f"  Combinaciones entidad-año: {len(por_ent_anio):,}".replace(",", "."))

# cumsum dentro de cada entidad: el valor de un año incluye todos los anteriores
por_ent_anio = por_ent_anio.sort_values(["nit_entidad", "anio"])
acumuladas = ["n", "adj", "oferentes", "oferentes_n", "precio", "precio_n"]
g = por_ent_anio.groupby("nit_entidad", observed=True)
for c in acumuladas:
    por_ent_anio[f"acum_{c}"] = g[c].cumsum()

# El perfil que le toca a un proceso del año A es el acumulado hasta A-1.
# Se logra corriendo el año una posición: el acumulado de 2020 se etiqueta
# como "el perfil válido para 2021".
por_ent_anio["anio_vigencia"] = por_ent_anio["anio"] + 1

perfil = pd.DataFrame({
    "nit_entidad":  por_ent_anio["nit_entidad"],
    "anio":         por_ent_anio["anio_vigencia"],
    "ent_procesos": por_ent_anio["acum_n"],
    "ent_tasa_adjudicacion": por_ent_anio["acum_adj"] / por_ent_anio["acum_n"],
    "ent_oferentes_prom": (por_ent_anio["acum_oferentes"]
                           / por_ent_anio["acum_oferentes_n"].replace(0, np.nan)),
    "ent_precio_base_medio": (por_ent_anio["acum_precio"]
                              / por_ent_anio["acum_precio_n"].replace(0, np.nan)),
})

print(f"  Filas del perfil con ventana: {len(perfil):,}".replace(",", "."))


# ══ 5 · SE PEGA TODO A LA TABLA BASE ════════════════════════════════════
antes = len(procesos)
procesos = procesos.merge(principal, left_on="id_del_portafolio",
                          right_index=True, how="left")
procesos = procesos.merge(cuantos, left_on="id_del_portafolio",
                          right_index=True, how="left")
procesos = procesos.merge(perfil, on=["nit_entidad", "anio"], how="left")
procesos = procesos.merge(ent_c, left_on="nit_entidad", right_index=True,
                          how="left")

# El NIT definitivo: el que registró la entidad en el proceso cuando está, y
# si no, el del proveedor al que efectivamente se le firmó el contrato.
procesos["nit_ganador"] = (procesos["nit_del_proveedor_adjudicado"]
                           .astype("string")
                           .fillna(procesos["nit_ganador_contrato"].astype("string")))

print(f"\n  Filas antes de unir : {antes:,}".replace(",", "."))
print(f"  Filas después       : {len(procesos):,}".replace(",", ".")
      + ("   correcto" if len(procesos) == antes
         else "   <-- LA UNIÓN DUPLICÓ FILAS"))


# ══ 6 · COBERTURA ═══════════════════════════════════════════════════════
print(f"\nCOBERTURA DEL ADJUDICATARIO")
print("-" * 78)
for nombre, m in [
    ("Solo desde Procesos",  procesos["nit_del_proveedor_adjudicado"].notna()),
    ("Solo desde Contratos", procesos["nit_ganador_contrato"].notna()),
    ("Combinando los dos",   procesos["nit_ganador"].notna()),
]:
    print(f"  {nombre:24} {int(m.sum()):>9,}".replace(",", ".")
          + f"   {m.mean()*100:>5.1f} %   " + "█" * int(m.mean() * 40))

adj = procesos["adjudicado_si"]
print(f"\n  CONTRASTE CONTRA LA VARIABLE OBJETIVO")
print(f"  De los {int(adj.sum()):,} marcados como adjudicados,".replace(",", ".")
      + f" tienen adjudicatario: "
      + f"{procesos.loc[adj, 'nit_ganador'].notna().mean()*100:.1f} %")
print(f"  De los {int((~adj).sum()):,} NO adjudicados,".replace(",", ".")
      + f" tienen adjudicatario: "
      + f"{procesos.loc[~adj, 'nit_ganador'].notna().mean()*100:.1f} %")
print(f"\n  El segundo porcentaje debería ser bajo: un proceso que no se")
print(f"  adjudicó no tendría por qué haber derivado en un contrato firmado.")
print(f"  Si sale alto, significa que la columna 'adjudicado' se queda sin")
print(f"  diligenciar en procesos que sí terminaron en contrato, y entonces")
print(f"  la variable objetivo hay que construirla combinando las dos señales.")

print(f"\nCOBERTURA DE LAS VARIABLES DE ENTIDAD")
print("-" * 78)
nuevas = ["ent_procesos", "ent_tasa_adjudicacion", "ent_oferentes_prom",
          "ent_precio_base_medio", "ent_contratos", "ent_valor_medio",
          "ent_reporta_pagos_pct", "ent_dias_adicionados"]
for c in nuevas:
    if c in procesos:
        p = procesos[c].notna().mean() * 100
        print(f"  {c:26} {p:>5.1f} %   " + "█" * int(p / 2.5))

print(f"\n  Las cuatro primeras están vacías en los procesos de 2017, que es")
print(f"  correcto: no hay año anterior con el cual construirles un perfil.")
vacio_2017 = procesos.loc[procesos["anio"] == 2017, "ent_procesos"].isna().mean()
print(f"  Vacías en 2017: {vacio_2017*100:.1f} %")


# ══ 7 · LA COMPROBACIÓN QUE IMPORTA ═════════════════════════════════════
# Si la tasa de adjudicación de la entidad separa procesos adjudicados de no
# adjudicados, la variable sirve. Si no separa, no aporta.
print(f"\n¿LA TASA HISTÓRICA DE LA ENTIDAD ANTICIPA EL RESULTADO?")
print("-" * 78)
con_perfil = procesos[procesos["ent_tasa_adjudicacion"].notna()].copy()
con_perfil["tramo"] = pd.cut(con_perfil["ent_tasa_adjudicacion"],
                             [0, .5, .7, .8, .9, .95, 1.01],
                             labels=["<50%", "50-70%", "70-80%", "80-90%",
                                     "90-95%", ">95%"])
tabla = (con_perfil.groupby("tramo", observed=True)
                   .agg(procesos=("id_del_proceso", "size"),
                        se_adjudico=("adjudicado_si", "mean")))
print(f"  {'tasa histórica de la entidad':30}{'procesos':>11}"
      f"{'se adjudicó':>13}")
print("-" * 78)
for tramo, fila in tabla.iterrows():
    barra = "█" * int(fila["se_adjudico"] * 40)
    print(f"  {str(tramo):30}{int(fila['procesos']):>11,}".replace(",", ".")
          + f"{fila['se_adjudico']*100:>12.1f} %  {barra}")

print(f"\n  Si el porcentaje de la derecha sube al bajar por la tabla, la")
print(f"  variable anticipa el resultado y entra al modelo. Si es plano,")
print(f"  no aporta y hay que decirlo.")

del por_ent_anio, perfil, ent_c, con_perfil, principal, cuantos
gc.collect()
print(f"\n  Memoria libre: {psutil.virtual_memory().available/1024**3:.1f} GB")


`CELDA 23`

## Paso 7 · Construir la variable objetivo y las variables derivadas

### 7.1 La variable objetivo, corregida

La primera intención era copiar la columna `adjudicado`. La revisión mostró que no
alcanza: **22.151 procesos figuran como no adjudicados y sin embargo derivaron en un
contrato firmado**. La casilla se queda sin diligenciar.

Un contrato firmado es evidencia material: alguien se ganó eso y se le está pagando.
Pesa más que una casilla vacía. Entonces la variable se construye combinando las dos
señales:

```
objetivo = 1  si la entidad lo marcó como adjudicado
              O si el proceso derivó en un contrato firmado
objetivo = 0  en cualquier otro caso
```

Con esa definición la tasa pasa del 82,5 % al 86,0 %, y esos 22.151 procesos dejan de
estar mal etiquetados.

**Lo que mide:** que el proceso llegó a término con alguien contratado. Lo contrario
—el 14 %— son los procesos que se declararon desiertos o se cancelaron: el esfuerzo
perdido que la empresa quiere evitar.

### 7.2 Las tres clases de variable, y por qué importa la diferencia

Esta es la parte que decide si el conjunto sirve para predecir o solo para describir.
Cada variable se clasifica según **cuándo se conoce**, mirado desde el momento en que
una empresa está decidiendo si prepara una propuesta.

| Clase | Cuándo se conoce | Ejemplos |
|---|---|---|
| **A · Al publicarse** | Está en el pliego desde el primer día | Entidad, modalidad, tipo de contrato, precio base, plazos, número de lotes, categoría, el perfil histórico de la entidad y la competencia inscrita en el nicho |
| **B · Mientras está abierto** | Se puede observar durante la convocatoria | Proveedores invitados, proveedores con invitación |
| **C · Solo al cerrar** | No existe antes del desenlace | Cuántas ofertas llegaron, quién ganó, por cuánto, el estado final, el contrato, las visualizaciones acumuladas |

**Las de clase C no pueden ser variables explicativas.** Un modelo que las use va a
acertar casi siempre en el cuaderno y va a fallar en la realidad, porque en la
realidad no están disponibles cuando hay que decidir. Se conservan en la tabla para
construir y verificar la variable objetivo, y van marcadas como prohibidas en el
diccionario.

**La clase B quedó en dos columnas, y es por una corrección.** Al principio tenía
cuatro. Una se botó porque está en cero en todas las filas: la casilla
`proveedores_que_manifestaron` existe en el formulario de la fuente pero no se
diligencia nunca. La otra, `visualizaciones_del`, se pasó a clase C: parecía
observable, pero el número que guarda la fuente es el total de visitas **al cerrarse**
el proceso, no las que había el día de la publicación, que son cero para todos. Es el
tipo de fuga más difícil de detectar, porque la variable no parece venir del futuro.

Las dos que quedan sí son legítimas: las invitaciones las manda la entidad al
publicar, así que se ven desde el primer día. Aun así entran marcadas aparte, para que
la fase siguiente decida si las usa.

El perfil del proveedor que ganó queda en clase C por definición: quién gana solo se
sabe cuando ya se adjudicó. Sirve para describir —"¿qué tipo de empresa gana en esta
entidad?"— pero nunca para predecir. Por eso las bases de Proveedores y Grupos no se
usan para perfilar al ganador, sino para contar cuánta competencia estaba inscrita en
el departamento y en la categoría antes de que el proceso se publicara.

### 7.3 Las variables que se construyen

Ninguna viene en los archivos. Todas se calculan a partir de lo que sí viene.

| Variable | Clase | Qué responde |
|---|---|---|
| `log_precio_base` | A | El tamaño del contrato, en escala comparable. Los precios van de millones a billones y sin logaritmo los grandes aplastan todo |
| `precio_vs_entidad` | A | Si este proceso es grande o pequeño **para esa entidad**. Mil millones es enorme en un municipio y rutinario en un ministerio |
| `duracion_dias` | A | El plazo llevado a una sola unidad. La fuente lo guarda en días, meses o años según el caso |
| `dias_para_cerrar` | A | Cuánto tiempo hay entre la publicación y el cierre. La mediana es de cinco días, que es muy poco para armar una propuesta, y un plazo muy corto a veces indica un proceso con destinatario previsto |
| `mes_publicacion` | A | En qué momento del año fiscal se publicó. Enero y diciembre no son lo mismo |
| `tiene_varios_lotes` | A | Si el proceso está partido. Los multilote se comportan distinto |

Hubo una séptima, `interes_por_vista`, que dividía los que manifestaron interés entre
los que miraron el proceso. Salió en cero en todas las filas porque el numerador está
en cero siempre. Se eliminó junto con la columna que la alimentaba.

### 7.4 El recorte de las cifras imposibles

Las columnas de dinero traen errores de captura de varios órdenes de magnitud: hay
valores de decenas de miles de billones de pesos, cuando el presupuesto nacional de un
año ronda los 500 billones.

El primer intento fue recortar al percentil 99,5 de cada columna. **No funcionó**, y la
razón vale la pena anotarla: en `ent_valor_medio` el percentil 99,5 era él mismo una
cifra absurda, porque una entidad con miles de procesos y un valor mal digitado
arrastra su propio percentil. El recorte dejó pasar todo.

Se pasó entonces a topes absolutos, calibrados contra una referencia real: un contrato
promedio de más de un billón de pesos no existe. Con ese criterio el recorte sí alcanzó
las filas que debía.

Se recorta en lugar de borrar las filas. La razón es que el resto de la fila sigue
siendo válido: el proceso existió, tuvo su entidad, su modalidad y su desenlace.
Perderlo entero por un campo mal digitado sería desperdiciar información buena.

La excepción son los plazos negativos. Ahí no se recorta, se deja vacío: un cierre
anterior a la publicación no es un plazo corto, es un par de fechas mal capturado, y
recortarlo a cero sería afirmar que el proceso se cerró el mismo día que se abrió. Un
vacío dice la verdad.

Todo recorte se declara, se cuenta cuántos valores afectó, y queda anotado en el
diccionario.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA 24  ·  Variable objetivo, recortes y variables derivadas
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────
# QUÉ ENTRA  : La tabla de procesos con el perfil de entidad ya pegado.
# QUÉ HACE   : Tres cosas. Construye la variable objetivo combinando la casilla
#              de la entidad con la evidencia del contrato firmado. Recorta las
#              cifras imposibles. Y calcula las variables derivadas, cada una
#              etiquetada según cuándo se conoce.
# QUÉ SALE   : La tabla analítica completa, con la clasificación de cada
#              columna en las tres clases A, B y C.
# PARA QUÉ   : Es el entregable de la fase. Lo que salga de aquí es lo que la
#              fase siguiente va a usar para modelar.
# ──────────────────────────────────────────────────────────────────────

# ══ 1 · LA VARIABLE OBJETIVO ════════════════════════════════════════════
print("CONSTRUYENDO LA VARIABLE OBJETIVO")
print("=" * 78)

marcado = procesos["adjudicado_si"]
con_contrato = procesos["nit_ganador_contrato"].notna()

procesos["objetivo"] = (marcado | con_contrato).astype("int8")

print(f"  {'Señal':44}{'procesos':>12}{'tasa':>9}")
print("-" * 78)
for nombre, serie in [
    ("Solo la casilla 'adjudicado'",           marcado),
    ("Solo la evidencia del contrato",         con_contrato),
    ("Las dos combinadas (variable objetivo)", procesos["objetivo"].astype(bool)),
]:
    print(f"  {nombre:44}{int(serie.sum()):>12,}".replace(",", ".")
          + f"{serie.mean()*100:>8.1f} %")

print("-" * 78)
print(f"  Rescatados por el contrato (la casilla estaba vacía): "
      f"{int(((~marcado) & con_contrato).sum()):,}".replace(",", "."))
print(f"  Marcados sin contrato encontrado: "
      f"{int((marcado & (~con_contrato)).sum()):,}".replace(",", ".")
      + "   (el contrato puede estar fuera del corte)")


# ══ 2 · RECORTE DE LAS CIFRAS IMPOSIBLES ════════════════════════════════
print(f"\nRECORTE DE LAS CIFRAS IMPOSIBLES")
print("=" * 78)
print("  Se usan topes absolutos y no percentiles. Un percentil se deja")
print("  arrastrar cuando los errores son muchos: si una entidad con miles de")
print("  procesos tiene mal digitado su valor, ese valor termina siendo su")
print("  propio percentil 99,5 y el recorte no toca nada. Un tope absoluto dice")
print("  que un contrato promedio de más de un billón de pesos no existe, y no")
print("  se deja convencer.")
print()

# Referencia para calibrar: el presupuesto general de la Nación ronda los 500
# billones de pesos al año. Un solo contrato no puede acercarse a esa cifra.
TOPES = {
    "precio_base":           (0, 5e12),    # 5 billones: megaproyecto de infraestructura
    "ent_valor_medio":       (0, 1e12),    # 1 billón como contrato promedio de una entidad
    "ent_precio_base_medio": (0, 1e12),
    "duracion_dias":         (0, 3650),    # diez años, el plazo máximo razonable
    "precio_vs_entidad":     (0, 100),     # cien veces el tamaño típico de su entidad
}


def recortar(s, nombre):
    """Lleva al borde los valores que no pueden existir, y reporta cuántos.

    Entra: una columna con cifras, algunas imposibles.
    Sale:  la misma columna con los extremos acotados.

    La conversión a decimal del principio no es cosmética: varias columnas
    quedaron guardadas como enteros para ahorrar memoria, y a un entero no se
    le puede poner un tope con decimales.

    No se borran filas. El proceso existió y el resto de sus datos sirve;
    perderlo entero por un campo mal digitado sería botar información buena.
    """
    piso, tope = TOPES[nombre]
    s = pd.to_numeric(s, errors="coerce").astype("float64")
    fuera = int(((s > tope) | (s < piso)).sum())
    validos = max(int(s.notna().sum()), 1)
    out = s.clip(piso, tope)
    print(f"  {nombre:24} tope {tope:>18,.0f}".replace(",", ".")
          + f"   recortó {fuera:>7,}".replace(",", ".")
          + f"   ({fuera/validos*100:.2f} %)")
    return out


procesos["precio_base_r"]     = recortar(procesos["precio_base"], "precio_base")
procesos["ent_valor_medio_r"] = recortar(procesos["ent_valor_medio"],
                                         "ent_valor_medio")
procesos["ent_precio_base_r"] = recortar(procesos["ent_precio_base_medio"],
                                         "ent_precio_base_medio")

# Los precios en cero se dejan como están. En acuerdos marco y en algunos
# procesos por lotes el valor no se fija al publicar, así que el cero es un
# dato real y no un error.
ceros = int((procesos["precio_base_r"] == 0).sum())
print(f"\n  Procesos con precio base cero: {ceros:,}".replace(",", ".")
      + f"   {ceros/len(procesos)*100:.2f} %   (acuerdos marco: se dejan)")


# ══ 3 · LAS VARIABLES DERIVADAS ═════════════════════════════════════════
print(f"\nVARIABLES DERIVADAS")
print("=" * 78)

# ── El tamaño, en escala comparable ──
procesos["log_precio_base"] = np.log1p(procesos["precio_base_r"].clip(lower=0))

# ── Grande o pequeño PARA ESA ENTIDAD ──
# Mil millones es enorme en un municipio y rutinario en un ministerio. Sin esta
# división el modelo solo vería el tamaño absoluto. Se recorta después de
# dividir, porque el cociente hereda los errores de las dos columnas.
procesos["precio_vs_entidad"] = recortar(
    procesos["precio_base_r"] / procesos["ent_precio_base_r"].replace(0, np.nan),
    "precio_vs_entidad")

# ── El plazo del contrato, llevado a días ──
# La fuente guarda la duración en la unidad que cada entidad escogió, así que
# el número solo no significa nada: 3 puede ser tres días o tres años.
A_DIAS = {"día(s)": 1, "dia(s)": 1, "días": 1, "dias": 1,
          "mes(es)": 30, "meses": 30, "mes": 30,
          "año(s)": 365, "años": 365, "ano(s)": 365,
          "semana(s)": 7, "semanas": 7,
          "hora(s)": 1 / 24}
factor = (procesos["unidad_de_duracion"].astype("string").str.strip().str.lower()
          .map(A_DIAS))
procesos["duracion_dias"] = recortar(
    pd.to_numeric(procesos["duracion"], errors="coerce").astype("float64") * factor,
    "duracion_dias")

# ── Cuánto tiempo se le dio al mercado para responder ──
# Un plazo corto entre la publicación y el cierre deja poco margen para
# preparar una propuesta, y a veces indica un proceso con destinatario previsto.
#
# Un plazo negativo no es un plazo corto: es un par de fechas mal capturado. Se
# deja vacío en vez de recortarlo a cero, porque cero diría que el proceso se
# cerró el mismo día que se abrió, y eso es afirmar algo que no se sabe. Un
# vacío dice la verdad: no hay dato. Los plazos de más de un año sí se
# recortan: una convocatoria abierta doce meses casi siempre es un proceso que
# se reabrió o una fecha mal digitada, pero el proceso en sí existió.
plazo = (procesos["fecha_de_recepcion_de"]
         - procesos["fecha_de_publicacion_del"]).dt.days.astype("float64")
print(f"  {'dias_para_cerrar':24} negativos {int((plazo < 0).sum()):>7,}"
      .replace(",", ".") + f"   de más de un año {int((plazo > 365).sum()):>5,}"
      .replace(",", "."))
procesos["dias_para_cerrar"] = plazo.where(plazo >= 0).clip(upper=365)

# ── En qué momento del año fiscal ──
procesos["mes_publicacion"] = procesos["fecha_de_publicacion_del"].dt.month

# ── Si está partido en lotes ──
procesos["tiene_varios_lotes"] = (pd.to_numeric(procesos["numero_de_lotes"],
                                                errors="coerce")
                                  .fillna(1) > 1).astype("int8")

derivadas = ["log_precio_base", "precio_vs_entidad", "duracion_dias",
             "dias_para_cerrar", "mes_publicacion", "tiene_varios_lotes"]

print(f"\n  {'variable':22}{'completa':>10}{'mínimo':>14}{'mediana':>14}{'máximo':>16}")
print("-" * 78)
for c in derivadas:
    s = pd.to_numeric(procesos[c], errors="coerce").astype("float64")
    print(f"  {c:22}{s.notna().mean()*100:>9.1f} %{s.min():>14,.2f}".replace(",", ".")
          + f"{s.median():>14,.2f}".replace(",", ".")
          + f"{s.max():>16,.2f}".replace(",", "."))

print(f"\n  La mediana de 'dias_para_cerrar' es el hallazgo que hay que")
print(f"  subrayar: la mitad de los procesos competitivos da ese plazo o menos")
print(f"  para presentar una propuesta. Para una empresa que tiene que reunir")
print(f"  documentos, cotizar y armar la oferta, ese margen es estrecho, y es")
print(f"  información directa para quien decide a qué proceso presentarse.")


# ══ 4 · LAS COLUMNAS QUE NO ENTRAN ══════════════════════════════════════
print(f"\nCOLUMNAS QUE NO ENTRAN Y POR QUÉ")
print("=" * 78)
for c in ("visualizaciones_del", "proveedores_que_manifestaron",
          "proveedores_invitados", "proveedores_con_invitacion"):
    if c in procesos.columns:
        s = pd.to_numeric(procesos[c], errors="coerce").astype("float64")
        print(f"  {c:30} en cero: {(s == 0).mean()*100:>5.1f} %"
              + f"   máximo: {s.max():>12,.0f}".replace(",", "."))

# "proveedores_que_manifestaron" está en cero en todas las filas: la casilla
# existe en el formulario de la fuente pero no se diligencia. Una columna
# constante no distingue nada, así que se bota.
if "proveedores_que_manifestaron" in procesos.columns:
    procesos.drop(columns=["proveedores_que_manifestaron"], inplace=True)

print()
print("  'proveedores_que_manifestaron' se bota: está en cero siempre.")
print()
print("  'visualizaciones_del' se queda, pero como clase C. Parece observable")
print("  mientras el proceso está abierto, y no lo es: el número que guarda la")
print("  fuente es el total de visitas al cerrarse el proceso, no las que había")
print("  el día de la publicación, que son cero para todos. Un modelo entrenado")
print("  con esa columna aprendería que los procesos muy vistos se adjudican:")
print("  es cierto y no sirve de nada, porque la empresa que está decidiendo si")
print("  se presenta no puede saber cuántas visitas tendrá el proceso dentro de")
print("  un mes. Es fuga de información con apariencia de variable observable,")
print("  que es la forma más difícil de detectar.")


# ══ 5 · LA CLASIFICACIÓN POR CLASES ═════════════════════════════════════
CLASE_A = [  # se conoce al publicarse el proceso
    "modalidad_de_contratacion", "tipo_de_contrato", "codigo_principal_de_categoria",
    "ordenentidad", "departamento_entidad", "ciudad_entidad", "fase",
    "unidad_de_duracion", "duracion", "numero_de_lotes", "precio_base",
    "precio_base_r", "log_precio_base", "precio_vs_entidad",
    "duracion_dias", "dias_para_cerrar", "mes_publicacion", "tiene_varios_lotes",
    "anio", "fecha_de_recepcion_de",
    "ent_procesos", "ent_tasa_adjudicacion", "ent_oferentes_prom",
    "ent_precio_base_medio", "ent_precio_base_r", "ent_contratos",
    "ent_valor_medio", "ent_valor_medio_r", "ent_reporta_pagos_pct",
    "ent_dias_adicionados",
]
CLASE_B = [  # se puede observar mientras el proceso está abierto
    "proveedores_invitados", "proveedores_con_invitacion",
]
CLASE_C = [  # solo se conoce al cerrar: PROHIBIDAS como explicativas
    "adjudicado", "adjudicado_si", "estado_del_procedimiento", "estado_resumen",
    "proveedores_unicos_con", "respuestas_al_procedimiento", "respuestas_externas",
    "conteo_de_respuestas_a_ofertas", "fecha_adjudicacion", "visualizaciones_del",
    "valor_total_adjudicacion", "nit_del_proveedor_adjudicado",
    "nit_ganador_contrato", "nit_ganador", "n_adjudicatarios", "n_contratistas",
    "valor_contrato_mayor", "fecha_primer_contrato",
]
LLAVES = ["id_del_proceso", "id_del_portafolio", "nit_entidad", "entidad",
          "codigo_entidad", "fecha_de_publicacion_del"]

print(f"\nCLASIFICACIÓN DE LAS COLUMNAS")
print("=" * 78)
for nombre, lista in [("A · al publicarse, SÍ se pueden usar", CLASE_A),
                      ("B · durante la convocatoria, con cuidado", CLASE_B),
                      ("C · solo al cerrar, PROHIBIDAS", CLASE_C),
                      ("Llaves y trazabilidad", LLAVES)]:
    presentes = [c for c in lista if c in procesos.columns]
    print(f"  {nombre:42} {len(presentes):>3} columnas")

sin_clasificar = [c for c in procesos.columns
                  if c not in CLASE_A + CLASE_B + CLASE_C + LLAVES
                  and c != "objetivo"]
if sin_clasificar:
    print(f"\n  SIN CLASIFICAR: {sin_clasificar}")
else:
    print(f"\n  Todas las columnas están clasificadas.")

print(f"\n  La tabla tiene {len(procesos):,} filas y {procesos.shape[1]} columnas"
      .replace(",", "."))
print(f"  Ocupa {procesos.memory_usage(deep=True).sum()/1024**2:.0f} MB")
print(f"  Memoria libre: {psutil.virtual_memory().available/1024**3:.1f} GB")
